# 99 · Final test evaluation of the 15 prompted LLMs
The only notebook that reads test labels. It scores the frozen zero-shot log-prob test batch `20261020_all_all_llms_batch_s42` (15 models × 7 tasks) under the protocol recorded in `docs/experiment-log.md` ("Final test-evaluation protocol", 2026-10-20) **before** any label was read.
- **Inputs:** `RUNS_ROOT/*-test_s42/` (config, predictions, complete.json), `data/processed/test/{cmsb,edos}_test.parquet` (labels), `configs/llms.yaml`.
- **Outputs:**
  - `results/final_eval/{tables,figures}/` and `results/final_eval/summary.json`
  - `OUTPUT_ROOT/{run_name}/`: config, environment, summary, README and the EXIST EvALL submissions (`evall_submissions/`; the manifest is also in `results/final_eval/tables/`)
- **Frozen rules:**
  - Soft refusal = answer mass < 0.5, scored wrong (strict). Sensitivity at {none, 0.1, 0.5, 0.9} is reported, never used to choose.
  - P(Yes)/mass ≥ 0.5 for binary/multilabel; argmax for multiclass. No dev tuning (no dev runs).
  - 95% bootstrap CIs (B = 10,000, seed 42) and paired tests against the best model with Holm correction.
- EXIST test is unlabelled, so EXIST appears only in the label-free sections and in the EvALL export.
- No raw post text is printed anywhere in this notebook.

In [ ]:
import os
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")
    PROJECT_ROOT = Path("/content/drive/MyDrive/Sexism-research")  # TODO: set to your Drive clone
else:
    PROJECT_ROOT = Path.cwd().parent  # local run from notebooks/

DATA_ROOT = PROJECT_ROOT / "data"
OUTPUT_ROOT = PROJECT_ROOT / "outputs"
RUNS_ROOT = OUTPUT_ROOT / "final_runs_result" / "outputs"  # the frozen 105-run test batch
RESULTS_ROOT = PROJECT_ROOT / "results"
HF_HOME = Path("/content/hf_cache") if IN_COLAB else PROJECT_ROOT / "hf_cache"
os.environ["HF_HOME"] = str(HF_HOME)
sys.path.insert(0, str(PROJECT_ROOT))
print("RUNS_ROOT   ", RUNS_ROOT, "\nRESULTS_ROOT", RESULTS_ROOT)

In [ ]:
# Same pins as 00 and requirements.txt; scipy/matplotlib are logged (Colab preinstalls them), not pinned yet
PINS = {"pandas": "3.0.6", "numpy": "2.5.4", "scikit-learn": "1.9.1", "pyarrow": "26.0.0", "pyyaml": "6.0.3"}
if IN_COLAB:
    PIP_ARGS = " ".join(f"{p}=={v}" for p, v in PINS.items())
    %pip install -q {PIP_ARGS} jsonschema
    print("If pip changed anything, restart the runtime, then run all again")

from importlib.metadata import version

off_pin = {p: version(p) for p in PINS if version(p) != PINS[p]}
assert not off_pin, f"installed {off_pin} differ from PINS: install the pins and restart the kernel"
print({p: version(p) for p in ["scipy", "matplotlib", "jsonschema"]})

In [ ]:
# Frozen 2026-10-20 in docs/experiment-log.md before any test label was read; do not edit after scoring
CONFIG = {
    "run_name": "20261020_all_all_llms_finaleval_s42",
    "batch": "20261020_all_all_llms_batch_s42",
    "split": "test",
    "backend": "logprob",
    "n_models": 15,
    "mass_threshold": 0.5,  # a priori soft-refusal cutoff (no dev runs to tune it)
    "sensitivity": [None, 0.1, 0.5, 0.9],
    "n_boot": 10_000,
    "seed": 42,
    "ci_level": 0.95,
    "alpha": 0.05,
    "ece_bins": 10,
    "tasks": [
        ("cmsb", "binary"), ("edos", "task_a"), ("edos", "task_b"), ("edos", "task_c"),
        ("exist", "task_1_1"), ("exist", "task_1_2"), ("exist", "task_1_3"),
    ],
    "labelled": [("cmsb", "binary"), ("edos", "task_a"), ("edos", "task_b"), ("edos", "task_c")],
    "evall_team": "HEFT",  # TODO: registered EvALL team name; no '_' allowed
}
T = CONFIG["mass_threshold"]
TABLES = RESULTS_ROOT / "final_eval" / "tables"
FIGS = RESULTS_ROOT / "final_eval" / "figures"
RUN_DIR = OUTPUT_ROOT / CONFIG["run_name"]
EVALL_DIR = RUN_DIR / "evall_submissions"  # ~13 MB, so kept out of results/; manifest goes to results/
for d in (TABLES, FIGS, EVALL_DIR, RUN_DIR):
    d.mkdir(parents=True, exist_ok=True)
TASK_NAME = {
    ("cmsb", "binary"): "CMSB binary", ("edos", "task_a"): "EDOS A", ("edos", "task_b"): "EDOS B",
    ("edos", "task_c"): "EDOS C", ("exist", "task_1_1"): "EXIST 1.1", ("exist", "task_1_2"): "EXIST 1.2",
    ("exist", "task_1_3"): "EXIST 1.3",
}

In [ ]:
import json
import re

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.stats import spearmanr
from sklearn.metrics import confusion_matrix, f1_score

from src.schemas.registry import get_schema
from src.utils import (
    answered_ids, apply_mass_threshold, bootstrap_indices, bootstrap_metric, ci_dotplot, compute_metrics,
    environment_info, evall_items, expected_calibration_error, fleiss_kappa, heatmap, holm, load_finished_runs,
    load_llm_registry, macro_f1_counts, mass_summary, mcnemar_exact, paired_bootstrap_p, percentile_ci,
    refusal_rates, save_table, set_seeds, set_style, sha256_file, strict_arrays, write_evall,
)

set_seeds(CONFIG["seed"])
set_style()

## 1 · Integrity audit (label-free)
Every (model, task) run must be finished, complete, duplicate-free, on the pinned revision, with one prompt version, and all models must cover the same post IDs per task.

In [ ]:
REGISTRY = load_llm_registry(PROJECT_ROOT / "configs" / "llms.yaml")
runs = [r for r in load_finished_runs(RUNS_ROOT, CONFIG["split"]) if r["backend"] == CONFIG["backend"]]


def n_params(spec) -> float:
    # Leading number of the model-card figure ("E4B" -> 4, effective params); used only for ordering and the size plot
    return float(re.search(r"[\d.]+", spec.params).group())


MODELS = sorted(REGISTRY, key=lambda m: (n_params(REGISTRY[m]), m))
DISPLAY = {m: REGISTRY[m].display_name or m for m in MODELS}  # names used in figures and paper tables
RUNS = {(r["dataset"], r["task"], r["model"]): r for r in runs}
audit = []
for ds, task in CONFIG["tasks"]:
    id_sets = []
    for m in MODELS:
        r = RUNS[(ds, task, m)]
        ids = [rec["id"] for rec in r["records"]]
        id_sets.append(frozenset(ids))
        audit.append({
            "dataset": ds, "task": task, "model": m, "run_name": r["run_name"], "n_examples": r["n_examples"],
            "n_pred": len(ids), "n_unique": len(set(ids)),
            "n_prompt_sha": len({rec["prompt_sha256"] for rec in r["records"]}),
            "revision_ok": r["revision"] == r["revision_requested"] == REGISTRY[m].revision,
            "thinking": r["thinking"], "mass_threshold_at_inference": r["mass_threshold"],
        })
    assert len(set(id_sets)) == 1, f"{ds}/{task}: models cover different IDs"
AUDIT = pd.DataFrame(audit)
assert len(AUDIT) == CONFIG["n_models"] * len(CONFIG["tasks"]) == 105
assert (AUDIT.n_pred == AUDIT.n_examples).all() and (AUDIT.n_unique == AUDIT.n_pred).all()
assert (AUDIT.n_prompt_sha == 1).all() and AUDIT.revision_ok.all() and not AUDIT.thinking.any()
AUDIT.to_csv(TABLES / "audit.csv", index=False)
ties = {(r["dataset"], r["task"], r["model"]): sum(sc["score"] == 0.5 for rec in r["records"] for sc in rec["scores"]) for r in runs}
print("label scores exactly 0.5 (ties at the threshold):", {k: v for k, v in ties.items() if v})
print(f"{len(AUDIT)} runs pass the audit; models by size: {MODELS}")
print(AUDIT.groupby(["dataset", "task"]).n_pred.first().to_dict())

## 2 · Answer mass and soft refusals (label-free, all 7 tasks)
The frozen cutoff is applied here, after inference: a label with answer mass < 0.5 is a soft refusal. Refusals are recorded per (post, label). A post counts as refused if any of its labels is.

In [ ]:
REC = {}  # (dataset, task, threshold) -> model -> thresholded records
for ds, task in CONFIG["tasks"]:
    for t in CONFIG["sensitivity"]:
        REC[(ds, task, t)] = {m: apply_mass_threshold(RUNS[(ds, task, m)]["records"], t) for m in MODELS}

mass_rows, label_rows = [], []
for ds, task in CONFIG["tasks"]:
    schema = get_schema(ds, task)
    for m in MODELS:
        recs = RUNS[(ds, task, m)]["records"]
        all_mass = np.array([s["mass"] for r in recs for s in r["scores"]])
        row = {"task": TASK_NAME[(ds, task)], "model": m, "mass_median": np.median(all_mass),
               "mass_p05": np.quantile(all_mass, 0.05), "mass_min": all_mass.min()}
        for t in CONFIG["sensitivity"]:
            n_ans = len(answered_ids(REC[(ds, task, t)][m], schema))
            row[f"refused_posts@{t}"] = 1 - n_ans / len(recs)
        mass_rows.append(row)
        for lab, rate in refusal_rates(recs, T).items():
            label_rows.append({"task": TASK_NAME[(ds, task)], "model": m, "label": lab, "refusal_rate": rate})
MASS = pd.DataFrame(mass_rows)
REFUSAL_LABEL = pd.DataFrame(label_rows)
save_table(MASS, TABLES, "mass_refusal", f"Answer mass and share of test posts with a soft refusal (mass < t) "
           f"per model and task; zero-shot log-prob, primary t = {T}.")
REFUSAL_LABEL.to_csv(TABLES / "refusal_per_label.csv", index=False)

ref_mat = MASS.pivot(index="model", columns="task", values=f"refused_posts@{T}").loc[MODELS, list(TASK_NAME.values())]
fig, ax = plt.subplots(figsize=(5.2, 4.4))
im = heatmap(ax, ref_mat.rename(index=DISPLAY), f"Share of test posts refused (answer mass < {T})")
fig.colorbar(im, ax=ax, shrink=0.7)
fig.savefig(FIGS / "refusal_heatmap.pdf")
plt.show()
print("Refused-post share at the primary cutoff, max over tasks:")
print(ref_mat.max(axis=1).sort_values(ascending=False).round(3).to_string())

In [ ]:
# Per-label refusal for the multilabel task (a model may answer one category and refuse another)
lab13 = REFUSAL_LABEL[REFUSAL_LABEL.task == "EXIST 1.3"].pivot(index="model", columns="label", values="refusal_rate")
print(lab13.loc[MODELS].round(3).to_string())

## 3 · Test labels (read here only)
CMSB uses the annotated-only setting the runs were made on (1,126 posts). EDOS B/C are scored on gold-sexist posts, as in SemEval-2023 Task 10. The B/C predictions on non-sexist posts are ignored. Only aggregate counts are printed.

In [ ]:
GOLD = {
    ds: pd.read_parquet(DATA_ROOT / "processed" / "test" / f"{ds}_test.parquet",
                        columns=["id", "sexist", "leak_group"] + (["category", "vector"] if ds == "edos" else []))
    .assign(id=lambda d: d["id"].astype(str))
    for ds in ("cmsb", "edos")
}
for ds, task in CONFIG["labelled"]:
    schema = get_schema(ds, task)
    pred_ids = {r["id"] for r in RUNS[(ds, task, MODELS[0])]["records"]}
    g = GOLD[ds][GOLD[ds].id.isin(pred_ids)]
    assert len(g) == len(pred_ids), f"{ds}: gold missing for some predicted IDs"
    vals = g[schema.column].dropna()
    if schema.task_type == "multiclass":
        assert set(vals.astype(str)) <= set(schema.labels()), f"{ds}/{task}: gold classes outside the schema"
    print(f"{TASK_NAME[(ds, task)]}: n gold = {len(vals)}, class counts = {vals.astype(str).value_counts().to_dict()}")
cmsb_ids = {r["id"] for r in RUNS[("cmsb", "binary", MODELS[0])]["records"]}
N_LEAK = int(GOLD["cmsb"][GOLD["cmsb"].id.isin(cmsb_ids)].leak_group.notna().sum())
print("CMSB leak_group rows: all test rows", int(GOLD["cmsb"].leak_group.notna().sum()), "| scored (annotated) posts", N_LEAK)

## 4 · Main results with 95% CIs and paired tests (CMSB, EDOS A/B/C)
Strict scoring: refused = wrong. Best model = highest point macro-F1. Each other model is compared with it by a paired bootstrap of the macro-F1 difference, plus exact McNemar on the binary tasks, with Holm correction over the 14 comparisons. The **fair subset** holds the posts that all 15 models answered (every label, at t = 0.5). A majority-class reference is shown for scale.

In [ ]:
IDX, ARR, MAIN = {}, {}, []
for ds, task in CONFIG["labelled"]:
    key, schema = (ds, task), get_schema(ds, task)
    recs = REC[(ds, task, T)]
    arr = {m: strict_arrays(GOLD[ds], recs[m], schema) for m in MODELS}
    ids, y, _, classes = arr[MODELS[0]]
    assert all(a[0] == ids and np.array_equal(a[1], y) for a in arr.values())
    K = len(classes)
    # Cross-check: the array path reproduces compute_metrics for every model
    for m in MODELS:
        ref = compute_metrics(GOLD[ds], recs[m], schema, split="test", backend="logprob", mass_threshold=T)
        assert abs(ref["macro_f1"] - macro_f1_counts(y, arr[m][2], K)) < 1e-12
    idx = bootstrap_indices(len(ids), CONFIG["n_boot"], CONFIG["seed"])
    boots = {m: bootstrap_metric(y, arr[m][2], K, idx) for m in MODELS}
    point = {m: macro_f1_counts(y, arr[m][2], K) for m in MODELS}
    best = max(MODELS, key=lambda m: point[m])
    fair = set.intersection(*(answered_ids(recs[m], schema) for m in MODELS))
    fair_mask = np.array([i in fair for i in ids])
    leak_mask = ~GOLD[ds].set_index("id").loc[ids, "leak_group"].notna().to_numpy()
    IDX[key], ARR[key] = idx, arr
    rows = []
    for m in MODELS:
        yp = arr[m][2]
        lo, hi = percentile_ci(boots[m], CONFIG["ci_level"])
        diff = boots[m] - boots[best]
        row = {
            "dataset": ds, "task": task, "setting": TASK_NAME[key], "model": m, "params": REGISTRY[m].params,
            "n_scored": len(ids), "n_refused": int(len(ids) - len(answered_ids(recs[m], schema) & set(ids))),
            "macro_f1": point[m], "ci_lo": lo, "ci_hi": hi, "accuracy": float((yp == y).mean()),
            "f1_positive": f1_score(y, yp, zero_division=0) if K == 2 else np.nan,
            "fair_n": int(fair_mask.sum()),
            "fair_macro_f1": macro_f1_counts(y[fair_mask], yp[fair_mask], K) if fair_mask.any() else np.nan,
            "macro_f1_no_leak": macro_f1_counts(y[leak_mask], yp[leak_mask], K) if ds == "cmsb" else np.nan,
            "delta_vs_best": point[m] - point[best],
            "delta_lo": percentile_ci(diff)[0] if m != best else np.nan,
            "delta_hi": percentile_ci(diff)[1] if m != best else np.nan,
            "p_boot": paired_bootstrap_p(diff) if m != best else np.nan,
            "p_mcnemar": mcnemar_exact(y, yp, arr[best][2])[2] if (K == 2 and m != best) else np.nan,
        }
        rows.append(row)
    others = [i for i, r in enumerate(rows) if r["model"] != best]
    for col in ("p_boot", "p_mcnemar"):
        if K == 2 or col == "p_boot":
            adj = holm([rows[i][col] for i in others])
            for i, a in zip(others, adj):
                rows[i][col + "_holm"] = a
    for r in rows:
        p_main = r.get("p_boot_holm")  # protocol: paired bootstrap of macro-F1; McNemar (accuracy) is secondary
        r["tied_with_best"] = r["model"] == best or (p_main is not None and p_main >= CONFIG["alpha"])
    maj = np.full_like(y, np.bincount(y, minlength=K).argmax())
    rows.append({"dataset": ds, "task": task, "setting": TASK_NAME[key], "model": "majority-class (reference)",
                 "n_scored": len(ids), "macro_f1": macro_f1_counts(y, maj, K), "accuracy": float((maj == y).mean())})
    MAIN += rows
MAIN = pd.DataFrame(MAIN)
cols = ["setting", "model", "params", "n_scored", "n_refused", "macro_f1", "ci_lo", "ci_hi", "accuracy", "f1_positive",
        "fair_n", "fair_macro_f1", "macro_f1_no_leak", "delta_vs_best", "delta_lo", "delta_hi", "p_boot_holm",
        "p_mcnemar_holm", "tied_with_best"]
MAIN = MAIN.reindex(columns=["dataset", "task"] + cols + ["p_boot", "p_mcnemar"])
MAIN.to_csv(TABLES / "main_results_full.csv", index=False)
for key in CONFIG["labelled"]:
    sub = MAIN[MAIN.setting == TASK_NAME[key]].drop(columns=["dataset", "task", "setting", "p_boot", "p_mcnemar"])
    sub = sub.dropna(axis=1, how="all")
    save_table(sub, TABLES, f"main_{key[0]}_{key[1]}",
               f"{TASK_NAME[key]} test, zero-shot log-prob, seed 42. Strict macro-F1 (refusal at mass < {T} "
               f"scored wrong) with 95\\% bootstrap CI (B={CONFIG['n_boot']}); fair = posts answered by all 15 models; "
               f"p-values Holm-adjusted vs the best model.")
show = ["setting", "model", "n_refused", "macro_f1", "ci_lo", "ci_hi", "fair_macro_f1", "delta_vs_best", "tied_with_best"]
print(MAIN[show].round(3).to_string(index=False))

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(11, 3.8), sharey=True)
for ax, key in zip(axes, CONFIG["labelled"]):
    sub = MAIN[(MAIN.setting == TASK_NAME[key]) & MAIN.model.isin(MODELS)].set_index("model").loc[MODELS].reset_index()
    maj = MAIN[(MAIN.setting == TASK_NAME[key]) & ~MAIN.model.isin(MODELS)].macro_f1.iat[0]
    ci_dotplot(ax, sub.assign(model=sub.model.map(DISPLAY)), TASK_NAME[key], ref=maj, ref_label="majority")
axes[0].legend(loc="lower left", bbox_to_anchor=(0, 1.08), ncol=2, frameon=False, fontsize=7)
fig.suptitle("")
fig.savefig(FIGS / "main_macro_f1_ci.pdf")
plt.show()

## 5 · Sensitivity of the primary metric to the refusal cutoff
Pre-declared grid. The primary number stays at t = 0.5 whatever this table shows. "none" = argmax over answer tokens regardless of mass.

In [ ]:
sens = []
for ds, task in CONFIG["labelled"]:
    schema = get_schema(ds, task)
    for m in MODELS:
        row = {"setting": TASK_NAME[(ds, task)], "model": m}
        for t in CONFIG["sensitivity"]:
            _, y, yp, classes = strict_arrays(GOLD[ds], REC[(ds, task, t)][m], schema)
            row[f"macro_f1@{t}"] = macro_f1_counts(y, yp, len(classes))
        sens.append(row)
SENS = pd.DataFrame(sens)
save_table(SENS, TABLES, "sensitivity_threshold",
           "Strict macro-F1 on test under each pre-declared soft-refusal cutoff (none = no cutoff); primary is 0.5.")
SENS["max_shift_vs_primary"] = SENS[[c for c in SENS if c.startswith("macro_f1@")]].sub(SENS[f"macro_f1@{T}"], axis=0).abs().max(axis=1)
print(SENS.groupby("setting").max_shift_vs_primary.describe()[["mean", "max"]].round(3))

## 6 · Error analysis (aggregate only)
Per-class F1 on EDOS B/C, prediction-share bias (predicted share − gold share per class; positive = over-predicted), and row-normalised confusion matrices for the three best models.

In [ ]:
for key in [("edos", "task_b"), ("edos", "task_c")]:
    classes = ARR[key][MODELS[0]][3]
    y = ARR[key][MODELS[0]][1]
    K = len(classes)
    per_class = pd.DataFrame(
        {m: f1_score(y, ARR[key][m][2], labels=list(range(K)), average=None, zero_division=0) for m in MODELS},
        index=classes).T
    gold_share = np.bincount(y, minlength=K) / len(y)
    bias = pd.DataFrame({m: np.bincount(ARR[key][m][2], minlength=K + 1)[:K] / len(y) - gold_share for m in MODELS},
                        index=classes).T
    per_class.to_csv(TABLES / f"per_class_f1_{key[1]}.csv")
    bias.to_csv(TABLES / f"pred_share_bias_{key[1]}.csv")
    fig, axes = plt.subplots(1, 2, figsize=(10, 4.4), sharey=True)
    im0 = heatmap(axes[0], per_class.rename(index=DISPLAY), f"{TASK_NAME[key]}: per-class F1 (strict)")
    lim = float(np.abs(bias.to_numpy()).max())
    im1 = heatmap(axes[1], bias.rename(index=DISPLAY), f"{TASK_NAME[key]}: predicted − gold share", kind="div", vmin=-lim, vmax=lim)
    fig.colorbar(im0, ax=axes[0], shrink=0.7)
    fig.colorbar(im1, ax=axes[1], shrink=0.7)
    fig.savefig(FIGS / f"per_class_{key[1]}.pdf")
    plt.show()

    top3 = MAIN[(MAIN.setting == TASK_NAME[key]) & MAIN.model.isin(MODELS)].nlargest(3, "macro_f1").model.tolist()
    short = [c.split("_")[0] for c in classes]  # "prejudiced_discussions" -> "prejudiced" for the axis
    fig, axes = plt.subplots(1, 3, figsize=(12, 4.2), sharey=True, layout="constrained")
    for ax, m in zip(axes, top3):
        cm = confusion_matrix(y, ARR[key][m][2], labels=list(range(K + 1)))[:K]
        cm = pd.DataFrame(cm / cm.sum(1, keepdims=True), index=short, columns=short + ["refused"])
        heatmap(ax, cm, DISPLAY[m], fmt="{:.2f}")
        ax.set_xlabel("predicted")
    for ax in axes[1:]:
        ax.tick_params(labelleft=False)
    axes[0].set_ylabel("gold (row-normalised)")
    fig.savefig(FIGS / f"confusion_top3_{key[1]}.pdf")
    plt.show()

pos = []
for key in [("cmsb", "binary"), ("edos", "task_a")]:
    y = ARR[key][MODELS[0]][1]
    for m in MODELS:
        pos.append({"setting": TASK_NAME[key], "model": m, "gold_pos_rate": y.mean(),
                    "pred_pos_rate": ARR[key][m][2].mean()})
POS = pd.DataFrame(pos)
save_table(POS, TABLES, "positive_rate", "Predicted vs gold sexist rate on test (strict predictions, t = 0.5).")
print(POS.pivot(index="model", columns="setting", values="pred_pos_rate").loc[MODELS].round(3).to_string())

## 7 · Calibration of P(Yes)/mass (binary tasks)
ECE uses 10 equal-width bins over all scored posts. The score is used whether or not the label was refused, since binary mass is near 1 for every model. `n_distinct` counts distinct score values, which shows the known bf16 coarse-grid issue (CLAUDE.md, known issues).

In [ ]:
cal, curves = [], {}
for key in [("cmsb", "binary"), ("edos", "task_a")]:
    ds = key[0]
    ids, y = ARR[key][MODELS[0]][0], ARR[key][MODELS[0]][1]
    for m in MODELS:
        by_id = {r["id"]: r["scores"][0]["score"] for r in RUNS[(ds, key[1], m)]["records"]}
        s = np.array([np.nan if by_id[i] is None else by_id[i] for i in ids])
        ok = ~np.isnan(s)
        ece, rows = expected_calibration_error(y[ok], s[ok], CONFIG["ece_bins"])
        curves[(key, m)] = rows
        cal.append({"setting": TASK_NAME[key], "model": m, "ece": ece, "n_distinct_scores": len(np.unique(s[ok])),
                    "share_score_exact_0.5": float((s[ok] == 0.5).mean()), "n_no_score": int((~ok).sum()),
                    "mean_score_minus_rate": float(s[ok].mean() - y[ok].mean())})
CAL = pd.DataFrame(cal)
save_table(CAL, TABLES, "calibration", "Expected calibration error (10 bins) of P(Yes)/mass on test; binary tasks.")
fig, axes = plt.subplots(1, 2, figsize=(7.5, 3.4), sharey=True)
for ax, key in zip(axes, [("cmsb", "binary"), ("edos", "task_a")]):
    top3 = MAIN[(MAIN.setting == TASK_NAME[key]) & MAIN.model.isin(MODELS)].nlargest(3, "macro_f1").model.tolist()
    ax.plot([0, 1], [0, 1], color="#6b6b67", linestyle="--", linewidth=0.9)
    for m, c, mk in zip(top3, ["#0d366b", "#3987e5", "#9ec5f4"], ["o", "s", "^"]):
        r = curves[(key, m)]
        ax.plot(r[:, 1], r[:, 2], color=c, marker=mk, markersize=4, linewidth=1.5, label=DISPLAY[m])
    ax.set_title(f"{TASK_NAME[key]}: reliability (top-3 models)", loc="left")
    ax.set_xlabel("mean P(Yes)/mass in bin")
    ax.legend(frameon=False, fontsize=7)
axes[0].set_ylabel("observed sexist rate")
fig.savefig(FIGS / "calibration_top3.pdf")
plt.show()
print(CAL.pivot(index="model", columns="setting", values="ece").loc[MODELS].round(3).to_string())

## 8 · Cross-model agreement (label-free, all 7 tasks)
Fleiss' κ across the 15 models on every test post. A refusal is its own category. For EXIST 1.3, κ is computed per label and then averaged. The pairwise percent-agreement heatmap is shown for EDOS A and EXIST 1.1.

In [ ]:
def codes_matrix(ds, task, t):
    # (posts x models) category codes; refusal = extra code K. Multilabel returns one matrix per label
    schema = get_schema(ds, task)
    recs = REC[(ds, task, t)]
    order = [r["id"] for r in recs[MODELS[0]]]
    if schema.task_type == "multilabel":
        labels = schema.labels()
        out = {}
        for j, lab in enumerate(labels):
            cols = []
            for m in MODELS:
                by_id = {r["id"]: r["scores"][j] for r in recs[m]}
                cols.append([2 if (s := by_id[i])["refused"] or s["pred"] is None else int(s["pred"]) for i in order])
            out[lab] = (np.array(cols).T, 3)
        return out
    classes = [False, True] if schema.task_type == "binary" else schema.labels()
    code = {c: k for k, c in enumerate(classes)}
    cols = []
    for m in MODELS:
        by_id = {r["id"]: r["scores"][0] for r in recs[m]}
        cols.append([len(classes) if (s := by_id[i])["refused"] or s["pred"] is None else code[s["pred"]] for i in order])
    return {None: (np.array(cols).T, len(classes) + 1)}

agree, PAIR = [], {}
for ds, task in CONFIG["tasks"]:
    mats = codes_matrix(ds, task, T)
    kappas = [fleiss_kappa(c, k) for c, k in mats.values()]
    agree.append({"task": TASK_NAME[(ds, task)], "fleiss_kappa": float(np.mean(kappas)),
                  "per_label": {str(l): round(k, 4) for l, k in zip(mats, kappas)} if len(kappas) > 1 else None})
    c = np.concatenate([c for c, _ in mats.values()], axis=0)
    PAIR[(ds, task)] = pd.DataFrame([[float((c[:, i] == c[:, j]).mean()) for j in range(len(MODELS))]
                                     for i in range(len(MODELS))], index=MODELS, columns=MODELS)
    PAIR[(ds, task)].to_csv(TABLES / f"pairwise_agreement_{ds}_{task}.csv")
AGREE = pd.DataFrame(agree)
AGREE.to_csv(TABLES / "fleiss_kappa.csv", index=False)
print(AGREE[["task", "fleiss_kappa"]].round(3).to_string(index=False))
fig, axes = plt.subplots(1, 2, figsize=(13, 5.6), sharey=True, layout="constrained")
for ax, key in zip(axes, [("edos", "task_a"), ("exist", "task_1_1")]):
    im = heatmap(ax, PAIR[key].rename(index=DISPLAY, columns=DISPLAY), f"{TASK_NAME[key]}: pairwise agreement", vmin=0.5, vmax=1)
axes[1].tick_params(labelleft=False)
fig.colorbar(im, ax=axes, shrink=0.6)
fig.savefig(FIGS / "pairwise_agreement.pdf")
plt.show()

## 9 · Model size vs macro-F1 (descriptive)
Size is the model-card figure (Gemma-4 E4B is plotted at 4, its effective size). With 15 models and one seed, Spearman ρ is descriptive only.

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(11, 2.9), sharey=False)
size_rows = []
for ax, key in zip(axes, CONFIG["labelled"]):
    sub = MAIN[(MAIN.setting == TASK_NAME[key]) & MAIN.model.isin(MODELS)]
    x = np.array([n_params(REGISTRY[m]) for m in sub.model])
    rho, p = spearmanr(x, sub.macro_f1)
    size_rows.append({"setting": TASK_NAME[key], "spearman_rho": rho, "p": p, "n_models": len(sub)})
    ax.scatter(x, sub.macro_f1, s=22, color="#256abf", zorder=3)
    ax.set_title(f"{TASK_NAME[key]} (ρ = {rho:.2f})", loc="left")
    ax.set_xlabel("params (B, model card)")
axes[0].set_ylabel("macro-F1 (strict)")
fig.savefig(FIGS / "size_vs_f1.pdf")
plt.show()
SIZE = pd.DataFrame(size_rows)
SIZE.to_csv(TABLES / "size_vs_f1.csv", index=False)
print(SIZE.round(3).to_string(index=False))

## 10 · EXIST label-free description and EvALL export
- EXIST test gold is not available, so scores will come from EvALL.
- Each model gets a hard and a soft file per task. A post with any refused label (mass < 0.5) is left out, which is strict: no answer.
- 1.3 soft `NO` = Π(1 − P(Yes)).
- The official validator is run on every file. Its count check expects 2,076 es+en items, so the English-only files are expected to fail that one check.

In [ ]:
import contextlib
import importlib.util
import io

desc = []
for ds, task in [k for k in CONFIG["tasks"] if k[0] == "exist"]:
    schema = get_schema(ds, task)
    for m in MODELS:
        recs = REC[(ds, task, T)][m]
        ans = [r for r in recs if r["id"] in answered_ids(recs, schema)]
        row = {"task": TASK_NAME[(ds, task)], "model": m, "n_answered": len(ans), "n_posts": len(recs)}
        if schema.task_type == "binary":
            row["pred_sexist_rate"] = np.mean([r["scores"][0]["pred"] for r in ans])
        elif schema.task_type == "multiclass":
            row |= {f"share_{c}": np.mean([r["scores"][0]["pred"] == c for r in ans]) for c in schema.labels()}
        else:
            row |= {f"rate_{s['label']}": np.mean([r["scores"][j]["pred"] for r in ans])
                    for j, s in enumerate(ans[0]["scores"])}
            row["rate_NO"] = np.mean([not any(s["pred"] for s in r["scores"]) for r in ans])
        desc.append(row)
EXIST_DESC = pd.DataFrame(desc)
EXIST_DESC.to_csv(TABLES / "exist_prediction_rates.csv", index=False)

spec = importlib.util.spec_from_file_location("exist_val", DATA_ROOT / "EXIST" / "evaluation" / "exist2025_format_val_V0.2.py")
val = importlib.util.module_from_spec(spec)
spec.loader.exec_module(val)
manifest = []
for ds, task in [k for k in CONFIG["tasks"] if k[0] == "exist"]:
    schema = get_schema(ds, task)
    for m in MODELS:
        for variant, t in (("strict", T), ("nocut", None)):
          recs = REC[(ds, task, t)][m]
          for soft in (False, True):
            items = evall_items(recs, schema, soft)
            run_id = m if variant == "strict" else f"{m}-nocut"
            path = write_evall(items, EVALL_DIR / task, task, soft, CONFIG["evall_team"], run_id)
            assert val.parser_json(str(path)), f"{path.name} fails the official JSON schema"
            assert all(600001 <= int(it["id"]) <= 600978 for it in items)
            manifest.append({"file": str(path.relative_to(RUN_DIR)), "run_name": RUNS[(ds, task, m)]["run_name"],
                             "model": m, "task": task, "type": "soft" if soft else "hard", "variant": variant,
                             "n_items": len(items), "n_omitted_refused": len(recs) - len(items), "mass_threshold": t,
                             "sha256": sha256_file(path)})
    buf = io.StringIO()
    with contextlib.redirect_stdout(buf):
        val.process_format_runs_by_task(str(EVALL_DIR / task) + "/")
    log = buf.getvalue()
    (EVALL_DIR / f"validator_log_{task}.txt").write_text(log)
    errs = [l for l in log.splitlines() if "ERROR" in l or "Error" in l]
    expected = [l for l in errs if "2076" in l or "Spanish" in l or "English dataset" in l]
    print(f"{task}: validator lines with errors = {len(errs)}; of which count checks (en-only) = {len(expected)}")
    assert len(errs) == len(expected), f"{task}: unexpected validator errors, see validator_log_{task}.txt"
MANIFEST = pd.DataFrame(manifest)
MANIFEST.to_csv(EVALL_DIR / "manifest.csv", index=False)
MANIFEST.to_csv(TABLES / "evall_manifest.csv", index=False)
print(MANIFEST.groupby(["task", "type"]).n_omitted_refused.agg(["min", "max"]).to_string())

## 11 · Paper tables (LaTeX, from the tables above)
Compact tables for the report and paper, in `results/final_eval/paper/`, built only from the frames computed above.

In [ ]:
from src.utils import tex_escape, write_paper_table

PAPER = RESULTS_ROOT / "final_eval" / "paper"
LAB = [TASK_NAME[k] for k in CONFIG["labelled"]]
f3 = lambda v: "--" if pd.isna(v) else f"{v:.3f}"
pct = lambda v: "0" if v == 0 else ("$<$0.1" if v < 0.001 else f"{100 * v:.1f}")
pv = lambda v: "--" if pd.isna(v) else ("$<$0.001" if v < 0.001 else f"{v:.3f}")
name = lambda m: tex_escape(DISPLAY[m])
LICENCE = {"apache-2.0": "Apache-2.0", "mit": "MIT", "llama3.1": "Llama 3.1 Community", "cc-by-nc-4.0": "CC-BY-NC-4.0",
           "glm-4": "GLM-4", "falcon-llm": "Falcon LLM", "exaone-1.2-nc": "EXAONE 1.2-NC",
           "nvidia-nemotron-open-model": "NVIDIA Open Model"}
row_of = lambda s, m: MAIN[(MAIN.setting == s) & (MAIN.model == m)].iloc[0]

write_paper_table(
    PAPER / "models.tex", ["Model", "Hugging Face repository", "Revision", "Params", "Licence"],
    [[name(m), rf"\texttt{{{tex_escape(REGISTRY[m].repo_id)}}}", rf"\texttt{{{REGISTRY[m].revision[:10]}}}",
      tex_escape(REGISTRY[m].params), LICENCE.get(REGISTRY[m].licence, tex_escape(REGISTRY[m].licence))] for m in MODELS],
    "Models, ordered by parameter count, with the repository revision used for every evaluation. Parameter counts "
    "are model-card figures; E4B denotes an effective size of 4B.", "tab:models", "lllll", star=True, fit=True)

# Main: macro-F1 and 95% CI; bold = best, dagger = not significantly different from the best (paired bootstrap, Holm)
rows = []
for m in MODELS:
    r = [name(m)]
    for s in LAB:
        x = row_of(s, m)
        best = x.delta_vs_best == 0
        v = rf"\textbf{{{x.macro_f1:.3f}}}" if best else f"{x.macro_f1:.3f}" + (r"$^{\dagger}$" if x.tied_with_best else "")
        r += [v, rf"{{\footnotesize[{x.ci_lo:.3f}, {x.ci_hi:.3f}]}}"]
    rows.append(r)
maj = MAIN[~MAIN.model.isin(MODELS)].set_index("setting")
rows.append([r"\textit{Majority class}"] + sum([[rf"\textit{{{maj.loc[s, 'macro_f1']:.3f}}}", ""] for s in LAB], []))
write_paper_table(
    PAPER / "main.tex", ["Model"] + ["Macro-F1", "95\\% CI"] * 4, rows,
    "Test macro-F1 under refusal-as-error scoring ($\\tau=0.5$; Section~\\ref{sec:protocol}). \\textbf{Bold}: best "
    "model per task; $^{\\dagger}$: not significantly different from the best model (paired bootstrap, Holm-adjusted, "
    "$\\alpha=0.05$). EDOS B/C are scored on the 970 sexist test posts.",
    "tab:main", "l" + "rl" * 4,
    group_header=r" & \multicolumn{2}{c}{CMSB} & \multicolumn{2}{c}{EDOS A} & \multicolumn{2}{c}{EDOS B} & "
                 r"\multicolumn{2}{c}{EDOS C} \\ \cmidrule(lr){2-3}\cmidrule(lr){4-5}\cmidrule(lr){6-7}\cmidrule(lr){8-9}",
    midrule_before=(len(MODELS),), star=True, fit=True)

# Significance against the best model: macro-F1 difference with paired-bootstrap Holm p, plus McNemar on binary tasks
rows = []
for m in MODELS:
    r = [name(m)]
    for s in LAB:
        x = row_of(s, m)
        if x.delta_vs_best == 0:
            r += ["best", "--"] + (["--"] if s in ("CMSB binary", "EDOS A") else [])
            continue
        r += [f"{x.delta_vs_best:+.3f}", pv(x.p_boot_holm)] + ([pv(x.p_mcnemar_holm)] if s in ("CMSB binary", "EDOS A") else [])
    rows.append(r)
write_paper_table(
    PAPER / "significance.tex", ["Model", "$\\Delta$", "$p_{\\text{boot}}$", "$p_{\\text{McN}}$", "$\\Delta$",
                                 "$p_{\\text{boot}}$", "$p_{\\text{McN}}$", "$\\Delta$", "$p_{\\text{boot}}$",
                                 "$\\Delta$", "$p_{\\text{boot}}$"], rows,
    "Comparison with the best model of each task. $\\Delta$: macro-F1 difference; $p_{\\text{boot}}$: paired-bootstrap "
    "test of the macro-F1 difference (primary); $p_{\\text{McN}}$: exact McNemar test of per-post correctness "
    "(secondary, binary tasks). All $p$-values are Holm-adjusted over the 14 comparisons of a task.",
    "tab:significance", "l" + "rrr" * 2 + "rr" * 2,
    group_header=r" & \multicolumn{3}{c}{CMSB} & \multicolumn{3}{c}{EDOS A} & \multicolumn{2}{c}{EDOS B} & "
                 r"\multicolumn{2}{c}{EDOS C} \\ \cmidrule(lr){2-4}\cmidrule(lr){5-7}\cmidrule(lr){8-9}\cmidrule(lr){10-11}",
    star=True, fit=True)

# Secondary metrics
rows = []
for m in MODELS:
    g = lambda s, c: row_of(s, m)[c]
    rows.append([name(m), f3(g("CMSB binary", "accuracy")), f3(g("CMSB binary", "f1_positive")),
                 f3(g("CMSB binary", "macro_f1_no_leak")), f3(g("EDOS A", "accuracy")), f3(g("EDOS A", "f1_positive")),
                 f3(g("EDOS B", "accuracy")), f3(g("EDOS B", "fair_macro_f1")), f3(g("EDOS C", "accuracy"))])
rows.append([r"\textit{Majority class}", f3(maj.loc["CMSB binary", "accuracy"]), "--", "--",
             f3(maj.loc["EDOS A", "accuracy"]), "--", f3(maj.loc["EDOS B", "accuracy"]), "--", f3(maj.loc["EDOS C", "accuracy"])])
nb_common = int(MAIN[MAIN.setting == "EDOS B"].fair_n.iloc[0])
write_paper_table(
    PAPER / "secondary.tex",
    ["Model", "Acc.", "F1$_{\\text{sexist}}$", "Macro-F1 (dedup.)", "Acc.", "F1$_{\\text{sexist}}$", "Acc.",
     "Macro-F1 (common)", "Acc."], rows,
    f"Secondary test metrics (refusal-as-error). Macro-F1 (dedup.): CMSB without the {N_LEAK} scored test posts whose "
    f"normalised text also occurs in the training or development split. Macro-F1 (common): EDOS B on the {nb_common} "
    "sexist posts answered by all 15 models (no such post exists for EDOS C).",
    "tab:secondary", "l" + "r" * 8,
    group_header=r" & \multicolumn{3}{c}{CMSB} & \multicolumn{2}{c}{EDOS A} & \multicolumn{2}{c}{EDOS B} & "
                 r"\multicolumn{1}{c}{EDOS C} \\ \cmidrule(lr){2-4}\cmidrule(lr){5-6}\cmidrule(lr){7-8}\cmidrule(lr){9-9}",
    midrule_before=(len(MODELS),), star=True, fit=True)

# Non-answers: scored posts for EDOS B/C (970 sexist posts); all 978 tweets for EXIST 1.2; all-post EDOS C for reference
ref_b = {m: row_of("EDOS B", m).n_refused / row_of("EDOS B", m).n_scored for m in MODELS}
ref_c = {m: row_of("EDOS C", m).n_refused / row_of("EDOS C", m).n_scored for m in MODELS}
ref_12 = MASS[MASS.task == "EXIST 1.2"].set_index("model")[f"refused_posts@{T}"]
ref_call = MASS[MASS.task == "EDOS C"].set_index("model")[f"refused_posts@{T}"]
any_ref = [m for m in MODELS if max(ref_b[m], ref_c[m], ref_12[m], ref_call[m]) > 0]
write_paper_table(
    PAPER / "refusal.tex", ["Model", "EDOS B", "EDOS C", "EXIST 1.2", "EDOS C (all 4{,}000)"],
    [[name(m), pct(ref_b[m]), pct(ref_c[m]), pct(ref_12[m]), pct(ref_call[m])] for m in any_ref],
    f"Non-answer rate (\\%, answer mass $<\\tau={T}$). EDOS B/C: share of the 970 scored sexist posts; EXIST 1.2: share "
    "of the 978 test tweets; last column: EDOS C over all test posts, including non-sexist ones. Models not listed, and "
    "all models on CMSB, EDOS A, EXIST 1.1 and every EXIST 1.3 label, have no non-answers.",
    "tab:refusal", "lrrrr")

# Sensitivity: only models whose EDOS B/C score changes across the grid
cols = [f"macro_f1@{t}" for t in CONFIG["sensitivity"]]
rows = []
for m in MODELS:
    vals = [SENS[(SENS.setting == s) & (SENS.model == m)].iloc[0][cols].to_numpy(dtype=float) for s in ["EDOS B", "EDOS C"]]
    if all(np.allclose(v, v[0]) for v in vals):
        continue
    rows.append([name(m)] + [f3(x) for v in vals for x in v])
write_paper_table(
    PAPER / "sensitivity.tex", ["Model"] + ["none", "0.1", "\\textbf{0.5}", "0.9"] * 2, rows,
    "Macro-F1 on EDOS B and C for each pre-declared non-answer threshold $\\tau$ (none: arg-max over the option letters "
    "regardless of mass; 0.5 is the primary setting). Models whose scores are identical for every $\\tau$ are omitted.",
    "tab:sensitivity", "l" + "r" * 8,
    group_header=r" & \multicolumn{4}{c}{EDOS B} & \multicolumn{4}{c}{EDOS C} \\ \cmidrule(lr){2-5}\cmidrule(lr){6-9}",
    star=True)

# Calibration and positive rate
rows = []
for m in MODELS:
    c = lambda s, k: CAL[(CAL.setting == s) & (CAL.model == m)].iloc[0][k]
    p = lambda s: POS[(POS.setting == s) & (POS.model == m)].iloc[0].pred_pos_rate
    rows.append([name(m)] + sum([[f3(p(s)), f3(c(s, "ece")), f"{c(s, 'mean_score_minus_rate'):+.3f}"]
                                 for s in ["CMSB binary", "EDOS A"]], []))
gold = POS.groupby("setting").gold_pos_rate.first()
rows.append([r"\textit{Gold}", f3(gold["CMSB binary"]), "--", "--", f3(gold["EDOS A"]), "--", "--"])
write_paper_table(
    PAPER / "calibration.tex", ["Model", "Pos.\\ rate", "ECE", "Gap", "Pos.\\ rate", "ECE", "Gap"], rows,
    "Predicted sexist rate against the gold rate, expected calibration error (ECE, 10 equal-width bins) of "
    "$P(\\text{Yes})/m$, and the signed gap between the mean score and the gold rate (positive: scores too high "
    "on average).", "tab:calibration", "lrrrrrr",
    group_header=r" & \multicolumn{3}{c}{CMSB} & \multicolumn{3}{c}{EDOS A} \\ \cmidrule(lr){2-4}\cmidrule(lr){5-7}",
    midrule_before=(len(MODELS),))

n_posts = {TASK_NAME[k]: len(RUNS[(k[0], k[1], MODELS[0])]["records"]) for k in CONFIG["tasks"]}
write_paper_table(
    PAPER / "agreement.tex", ["Task", "Classes", "Posts", "Fleiss' $\\kappa$"],
    [[tex_escape(r.task), {"EDOS B": "4", "EDOS C": "11", "EXIST 1.2": "4", "EXIST 1.3": "5 $\\times$ 2"}.get(r.task, "2"),
      f"{n_posts[r.task]:,}".replace(",", "{,}"), f3(r.fleiss_kappa)] for r in AGREE.itertuples()],
    "Agreement among the 15 models' test predictions, over all test posts of each task (a non-answer is its own "
    "category; EXIST 1.3: mean over the five binary labels). Values are not directly comparable across tasks with "
    "different numbers of categories.", "tab:agreement", "lrrr")
print(sorted(p.name for p in PAPER.glob("*.tex")))

In [ ]:
# Prompt appendix: the exact system and user messages, with a placeholder post; Yes/No tasks show every label's question
from src.prompts.builder import PromptTemplate


def tex_verb(s: str) -> str:
    # Escape for an upright typewriter block; keeps line breaks
    for a, b in (("\\", r"\textbackslash{}"), ("{", r"\{"), ("}", r"\}"), ("$", r"\$"), ("&", r"\&"), ("%", r"\%"),
                 ("_", r"\_"), ("#", r"\#"), ("^", r"\^{}"), ("~", r"\~{}")):
        s = s.replace(a, b)
    # Blank lines become paragraph breaks; a forced break on an empty line is a LaTeX error
    return s.replace("\n\n", "\x00").replace("\n", "\\\\\n").replace("\x00", "\n\n")


blocks = []
for ds, task in CONFIG["tasks"]:
    schema = get_schema(ds, task)
    tpl = PromptTemplate.load(schema)
    post = "<post text>"
    if schema.task_type == "multiclass":
        msgs = [tpl.option_messages(post)]
        letters = ", ".join(f"{l} = {c}" for l, c in zip(tpl.option_letters(), schema.labels()))
        note = f"Answer letters: {letters}."
    else:
        msgs = [tpl.yes_no_messages(post, lab) for lab in schema.labels()]
        note = "One question per label; answers read from the Yes/No tokens."
    body = [rf"\paragraph{{{tex_escape(TASK_NAME[(ds, task)])}.}} System message: ``{tex_verb(tpl.system)}'' {tex_escape(note)}"]
    for msg in msgs:
        body.append(r"\begin{quote}\footnotesize\ttfamily\raggedright " + tex_verb(msg[1]["content"]) + r"\end{quote}")
    blocks.append("\n".join(body))
(PAPER / "prompts.tex").write_text("\n\n".join(blocks) + "\n")
print("prompts.tex:", len(blocks), "tasks")

## 12 · Save

In [ ]:
summary = {
    "run_name": CONFIG["run_name"], "batch": CONFIG["batch"], "config": CONFIG, "models": MODELS,
    "protocol": "docs/experiment-log.md, 2026-10-20 'Final test-evaluation protocol' (frozen before labels were read)",
    "main": MAIN.replace({np.nan: None}).to_dict(orient="records"),
    "fleiss_kappa": AGREE.replace({np.nan: None}).to_dict(orient="records"),
    "size_vs_f1": SIZE.to_dict(orient="records"),
    "runs": AUDIT[["dataset", "task", "model", "run_name"]].to_dict(orient="records"),
}
(RESULTS_ROOT / "final_eval" / "summary.json").write_text(json.dumps(summary, indent=1, default=str))
(RUN_DIR / "summary.json").write_text(json.dumps(summary, indent=1, default=str))
(RUN_DIR / "config.json").write_text(json.dumps(CONFIG, indent=2, default=str))
(RUN_DIR / "environment.json").write_text(json.dumps(environment_info(PROJECT_ROOT), indent=1))
(RUN_DIR / "README.md").write_text(
    f"# {CONFIG['run_name']}\n\n"
    f"**Purpose.** Final test scoring of `{CONFIG['batch']}` (15 LLMs × 7 tasks, zero-shot log-prob).\n"
    "**Notebook.** `notebooks/99_final_eval.ipynb`.\n"
    f"**Config.** Soft refusal at mass < {T} (strict), sensitivity {CONFIG['sensitivity']}, "
    f"bootstrap B={CONFIG['n_boot']} seed {CONFIG['seed']}, Holm α={CONFIG['alpha']}.\n"
    "**Outcome.** Tables/figures in `results/final_eval/`, EvALL files in `evall_submissions/` here; "
    "EXIST scores pending EvALL.\n")
print("saved", RUN_DIR, "and", RESULTS_ROOT / "final_eval")

## Summary of findings (test, zero-shot log-prob, seed 42, strict scoring at mass < 0.5)
Observations only. Sources: `results/final_eval/tables/*`. Per-model run names are in `audit.csv` (`20261020_{dataset}_{task}_{model}_logprob-zs-test_s42`). These are one seed and one prompt per task, with no dev tuning, so differences between models describe this setting only.

**Integrity.** All 105 runs are complete, have no duplicate IDs, run on the pinned revisions and use one prompt version each (`audit.csv`).

**Refusals** (`mass_refusal.*`, `refusal_heatmap.pdf`):
- Binary and Yes/No tasks: soft refusals are 0% for every model (CMSB, EDOS A, EXIST 1.1, EXIST 1.3 per label).
- Lettered-option tasks concentrate them:
  - EDOS C, on the 970 scored sexist posts: Aya-Expanse-8B 98.8%, SmolLM3-3B 97.4%, Granite-4.1-8B 80.9%, Ministral-3-3B 7.3%, Gemma-4-E4B 0.6%.
  - EDOS C over all 4,000 posts, Gemma-4-E4B is 13.5%: its non-answers fall mostly on non-sexist posts.
  - EXIST 1.2: Granite 17%.
- So the fair subset is empty for EDOS C (no post is answered by all 15 models). For EDOS B it holds 865 of 970 gold-sexist posts.

**Main results** (`main_*.tex`, `main_macro_f1_ci.pdf`). Point estimates with 95% bootstrap CIs; "tied" = not significantly different from the best model (paired bootstrap of macro-F1, Holm, α = .05; McNemar is secondary, in `significance.tex`).

| Setting | Best model, macro-F1 [95% CI] | Tied with best | Median of 15 | Majority-class |
|---|---|---|---|---|
| CMSB binary | Granite 4.1-8B, 0.821 [0.797, 0.843] | Llama-3.1-8B, Qwen3.5-4B, Gemma-4 E4B, Jamba2-3B, SmolLM3-3B | 0.767 | 0.405 |
| EDOS A | Gemma-4 E4B, 0.712 [0.696, 0.726] | Jamba2-3B, SmolLM3-3B, Qwen3.5-4B | 0.577 | 0.431 |
| EDOS B | Falcon-H1-7B, 0.508 [0.466, 0.549] | none | 0.353 | 0.159 |
| EDOS C | Gemma-4 E4B, 0.256 [0.227, 0.284] | Falcon-H1-7B (0.253) | 0.107 | 0.032 |

- No single model is best on every task.
- On EDOS C, 4 models (Aya, SmolLM3, Jamba2, EXAONE-4) score near or below the majority-class reference.

**Leak rows.** 33 CMSB test rows carry a `leak_group`, but only 3 of them are among the 1,126 scored posts. Excluding those 3 changes macro-F1 by at most 0.002 (`macro_f1_no_leak`).

**Cutoff sensitivity** (`sensitivity_threshold.*`):
- Between none and 0.5, no CMSB or EDOS A score changes (max shift 0.000).
- The large moves are on EDOS C: Aya 0.116 → 0.003, SmolLM3 0.109 → 0.009, Granite 0.171 → 0.107. So these models are reported with both numbers.
- At 0.9, SmolLM3, Ministral-3 and Apertus fall to 0 on EDOS B and C. Granite drops on EDOS C (0.107 → 0.016) and on EDOS B (0.362 → 0.271). On the binary tasks only Ministral-3 changes (CMSB 0.726 → 0.714, EDOS A 0.502 → 0.454).

**Bias and calibration** (`positive_rate.*`, `calibration.*`):
- Most models over-predict the sexist class. Gold rate is 0.32 on CMSB (14 of 15 models predict more positives) and 0.24 on EDOS A (all 15 do; 0.26–0.76).
- ECE of P(Yes)/mass: CMSB 0.03–0.33, EDOS A 0.09–0.51.
- No exact 0.5 scores on the labelled tasks; one EXIST 1.3 label score (Gemma-4-E4B) is exactly 0.5 and is predicted positive.
- EDOS B: animosity is often predicted as derogation; 9 of 15 models never predict prejudiced discussions, which is also the last option (`confusion_top3_task_b.pdf`).
- EDOS C: several models put most predictions on one vector, e.g. Jamba2 on 2.1 and EXAONE-4 on 2.3 (`per_class_task_c.pdf`).

**Agreement** (`fleiss_kappa.csv`): Fleiss' κ across the 15 models is 0.57 (CMSB), 0.43 (EDOS A), 0.20 (EDOS B), 0.05 (EDOS C), 0.49 (EXIST 1.1), 0.12 (EXIST 1.2) and 0.44 (EXIST 1.3, mean over labels).

**Size** (`size_vs_f1.csv`): Spearman ρ between model-card size and macro-F1 is 0.37 (CMSB), −0.10 (EDOS A), 0.52 (EDOS B, p = .045, uncorrected) and 0.27 (EDOS C). With n = 15 and one seed this is descriptive only.

**EXIST.** No local scores, since the test gold is unavailable. The 180 EvALL files (strict, with refused posts left out, plus no-cutoff variants) pass the official JSON-schema check. Only the validator's count checks fail, as expected (English only, 978 of 2,076 items). Refused posts are left out: EXIST 1.2 Granite 164, SmolLM3 25, Ministral 11.

**Open before the paper.**
- EvALL submission: confirm the team name and that English-only runs are accepted.
- Qwen3.5-4B thinking-on run.
- State in the paper that thresholds and the refusal cutoff were fixed a priori (no dev runs).
- Pin scipy/matplotlib.

## Post-hoc analyses (addendum A1, 2026-10-23)
Everything below follows `docs/12-protocol-addendum.md`, which fixed every rule before these cells ran. Sections 1–12 above are frozen and unchanged; rerunning them must reproduce `results/final_eval/` tables byte-for-byte.
- **Labels:** PH = post-hoc secondary analysis of the frozen predictions (test labels were read on 2026-10-20); PR = pre-registered new runs. Every table carries its label in `results/posthoc_eval/README.md`.
- **Inputs:** the frozen batch through its experiment manifest (`OUTPUT_ROOT/experiments/20261020-000000_experiment_zeroshot-frozen`, registered read-only below), and each new experiment named in `PH["experiments"]` (cells skip a batch that has not run yet).
- **Outputs:** `results/posthoc_eval/{tables,figures}/` and `OUTPUT_ROOT/20261023_all_all_llms_posthoc_s42/` (per-model failure tables, XAI id lists, EvALL files; ids and codes only, no text).
- **Sanity mode** (`PH["sanity"] = True`): B = 200, may read `-sample` experiments, writes only under `OUTPUT_ROOT/…-sanity/`, never into `results/`. Use it to check the pipeline end to end; its numbers are not results.
- No raw post text is printed.

In [ ]:
# Post-hoc setup: config, the frozen batch via its manifest, and per-post scores shared by the cells below
if IN_COLAB:  # PyEvALL pins numpy 1.26 / pandas 2.2, so it goes in without its dependencies (docs/12 §3.2)
    %pip install -q --no-deps pyevall==0.2.11 tabulate==0.9.0 jsbeautifier==1.14.9 editorconfig==0.17.1
import itertools

from scipy.stats import kendalltau

from src.prompts import PromptTemplate
from src.utils import (
    HIERARCHY, approx_randomization_p, batch_calibrated, binary_metrics, contrast_metrics, contrast_pairs,
    failure_table, find_experiment, fit_platt, fit_temperature, fit_threshold, format_p, gold_subset, label_scores,
    length_bin, load_experiment_runs, load_lexicons, mde, multilabel_macro_f1, per_class_f1, pride_debias,
    pride_prior, prior_matched, pyevall_scores, rank_stats, rate_table, register_experiment, risk_coverage,
    simultaneous_ci, soft_losses, spearman_ci, word_hits,
)

PH = {
    "run_name": "20261023_all_all_llms_posthoc_s42",
    "addendum": "docs/12-protocol-addendum.md",
    "sanity": False,  # True: B = 200, reads '-sample' experiments, writes only under OUTPUT_ROOT/...-sanity
    # Experiment folders under OUTPUT_ROOT/experiments (a name or a list); None until that batch has run (skipped)
    "experiments": {
        "frozen": "20261020-000000_experiment_zeroshot-frozen", "devsweep": None, "format_check": None,
        "answer_prefix": None, "option_order": None, "prompt5": None, "encoders": None, "xai": None,
    },
    "evall_confirmed": False,  # True only after the organisers confirm English-only scoring (docs/12 §2.9)
    "evall_team": "TODO",  # registered EvALL team name, no '_' (the frozen export above keeps its own setting)
    "n_boot": 10_000, "n_boot_small": 2_000, "seed": 42, "m_comparisons": 14,
    "train_prior": {"cmsb": 0.3204, "edos": 0.2427, "exist": 0.3962},  # docs/12 §1
    "xai_per_cell": 25,  # posts per TP/FP/FN/TN cell for the explanation sample (docs/12 §3.8)
}
SAN = PH["sanity"]
NB = 200 if SAN else PH["n_boot"]
NB_S = 200 if SAN else PH["n_boot_small"]
PH_RUN = PH["run_name"] + ("-sanity" if SAN else "")
PH_DIR = OUTPUT_ROOT / PH_RUN
PH_RES = PH_DIR / "results" if SAN else RESULTS_ROOT / "posthoc_eval"
PH_TABLES, PH_FIGS = PH_RES / "tables", PH_RES / "figures"
for d in (PH_DIR, PH_TABLES, PH_FIGS):
    d.mkdir(parents=True, exist_ok=True)
EXP_ROOT = OUTPUT_ROOT / "experiments"
PH_FILES = []  # (table name, status label, one-line description) for the README


def ph_save(df, name, label, note):
    """Write a post-hoc table and record it, with its PH/PR label, for the README."""
    df = pd.DataFrame(df)
    df.to_csv(PH_TABLES / f"{name}.csv", index=False)
    PH_FILES.append((name, label, note))
    return df


def exp_runs(key, split, kind="llm"):
    """Finished runs of the experiment(s) named in PH['experiments'] (a name or a list), via manifests; None if not run."""
    names = PH["experiments"].get(key)
    if names is None:
        return None
    names = [names] if isinstance(names, str) else names
    return load_experiment_runs([EXP_ROOT / n for n in names], split, kind, allow_sample=SAN)


# Register the frozen batch once (read-only; no run file is moved or changed); the manifest must give the same inputs
if find_experiment(OUTPUT_ROOT, "zeroshot-frozen") is None:
    register_experiment(
        OUTPUT_ROOT, "zeroshot-frozen", RUNS_ROOT, created="20261020-000000",
        purpose="Frozen zero-shot log-prob test batch (15 LLMs x 7 tasks), scored in sections 1-12 of 99",
        protocol_ref="docs/06-final-evaluation.md (frozen 2026-10-20)",
        config={"batch": CONFIG["batch"], "notebook": "notebooks/20_llm_runs.ipynb"},
    )
FROZEN = {(r["dataset"], r["task"], r["model"]): r for r in exp_runs("frozen", "test") if r["backend"] == "logprob"}
assert {k: r["run_name"] for k, r in FROZEN.items()} == {k: r["run_name"] for k, r in RUNS.items()}
assert all(FROZEN[k]["records"] == RUNS[k]["records"] for k in RUNS), "manifest inputs differ from sections 1-12"

BIN = [k for k in CONFIG["labelled"] if get_schema(*k).task_type == "binary"]
MC = [k for k in CONFIG["labelled"] if get_schema(*k).task_type == "multiclass"]
LABEL = {k: get_schema(*k).labels()[0] for k in CONFIG["tasks"] if get_schema(*k).task_type == "binary"}
SC = {(ds, task, m): label_scores(RUNS[(ds, task, m)]["records"], get_schema(ds, task))
      for ds, task in CONFIG["tasks"] for m in MODELS}
TRAIN_SHARES = {}  # train class shares among gold-sexist train posts (docs/12 §1)
for key in MC:
    col = get_schema(*key).column
    tr = pd.read_parquet(DATA_ROOT / "processed" / "train" / "edos_train.parquet", columns=[col])[col].dropna()
    TRAIN_SHARES[key] = tr.value_counts(normalize=True).reindex(get_schema(*key).labels()).fillna(0).to_numpy()
MAIN_BY = {k: MAIN[(MAIN.dataset == k[0]) & (MAIN.task == k[1]) & MAIN.model.isin(MODELS)].set_index("model")
           for k in CONFIG["labelled"]}
BEST = {k: MAIN_BY[k].macro_f1.idxmax() for k in CONFIG["labelled"]}
print(PH_RUN, "| B =", NB, "| frozen runs via manifest:", len(FROZEN), "| best by primary macro-F1:", BEST)

In [ ]:
# 2.1 Statistics display (PH): simultaneous intervals, MDE, rank uncertainty, AR check, size and cross-task agreement
BOOTS, rows, mde_rows = {}, [], []
for key in CONFIG["labelled"]:
    ids, y, _, classes = ARR[key][MODELS[0]]
    K, idx, best, main = len(classes), IDX[key][:NB], BEST[key], MAIN_BY[key]
    BOOTS[key] = {m: bootstrap_metric(y, ARR[key][m][2], K, idx) for m in MODELS}
    ranks = rank_stats(BOOTS[key], CONFIG["ci_level"])
    task_rows = []
    for m in MODELS:
        r = {"setting": TASK_NAME[key], "model": m, "macro_f1": main.loc[m, "macro_f1"], **ranks[m]}
        if m != best:
            diff = BOOTS[key][m] - BOOTS[key][best]
            lo, hi = simultaneous_ci(diff, CONFIG["ci_level"], PH["m_comparisons"])
            r |= {
                "delta": main.loc[m, "delta_vs_best"], "delta_lo95": main.loc[m, "delta_lo"],
                "delta_hi95": main.loc[m, "delta_hi"], "delta_lo_simult": lo, "delta_hi_simult": hi,
                "tied_frozen": bool(main.loc[m, "tied_with_best"]),
                "ci95_excludes_0": bool(main.loc[m, "delta_hi"] < 0 or main.loc[m, "delta_lo"] > 0),
                "simult_excludes_0": bool(hi < 0 or lo > 0),
                "p_boot_holm": format_p(main.loc[m, "p_boot_holm"], CONFIG["n_boot"]),
                "mde": mde(diff), "mde_holm_first": mde(diff, m=PH["m_comparisons"]),
                "p_ar": approx_randomization_p(y, ARR[key][m][2], ARR[key][best][2], K, NB, PH["seed"]),
            }
        task_rows.append(r)
    others = [r for r in task_rows if r["model"] != best]
    for r, a in zip(others, holm([r["p_ar"] for r in others])):
        r |= {"p_ar_holm": a, "ar_would_change_tie": bool((a >= CONFIG["alpha"]) != r["tied_frozen"])}
    rows += task_rows
    mde_rows.append({"setting": TASK_NAME[key], "n": len(ids), "mde_median": np.median([r["mde"] for r in others]),
                     "mde_holm_first_median": np.median([r["mde_holm_first"] for r in others])})
STATS = ph_save(rows, "stats_display", "PH", "Δ vs best with per-comparison and simultaneous (Bonferroni /14) "
                "95% intervals, MDE, P(best), rank interval, approximate-randomization p (docs/12 §2.1)")
ph_save(mde_rows, "mde_by_task", "PH", "Median minimum detectable macro-F1 difference vs the best model (80% power)")

size = []
for key in CONFIG["labelled"]:
    main = MAIN_BY[key]
    rho, lo, hi, p = spearman_ci(np.array([n_params(REGISTRY[m]) for m in main.index]), main.macro_f1.to_numpy())
    size.append({"setting": TASK_NAME[key], "n_models": len(main), "rho": rho, "ci_lo": lo, "ci_hi": hi, "p": p})
for r, a in zip(size, holm([r["p"] for r in size])):
    r["p_holm"] = a
ph_save(size, "size_vs_f1_ci", "PH", "Spearman ρ of model size vs macro-F1, Fisher-z 95% CI, Holm over 4 tasks")

tau = []
for a, b in itertools.combinations(CONFIG["labelled"], 2):
    fa, fb = [MAIN_BY[a].loc[m, "macro_f1"] for m in MODELS], [MAIN_BY[b].loc[m, "macro_f1"] for m in MODELS]
    draws = [kendalltau([BOOTS[a][m][i] for m in MODELS], [BOOTS[b][m][i] for m in MODELS])[0] for i in range(NB_S)]
    t0, p0 = kendalltau(fa, fb)
    tau.append({"task_a": TASK_NAME[a], "task_b": TASK_NAME[b], "kendall_tau": t0, "p": p0,
                "ci_lo": np.quantile(draws, 0.025), "ci_hi": np.quantile(draws, 0.975)})
ph_save(tau, "cross_task_tau", "PH", "Kendall τ_b of model rankings between tasks, bootstrap 95% CI (B=2,000)")
flag = STATS[STATS.get("ci95_excludes_0", False).fillna(False).astype(bool) & STATS.get("tied_frozen", False).fillna(False).astype(bool)]
print(f"{len(flag)} frozen 'not separable' marks sit next to a per-comparison 95% Δ interval that excludes 0 "
      "(the dagger is family-wise; see stats_display.csv)")

In [ ]:
# 2.2 Threshold-free and proper-score metrics (PH): AUROC/AUPRC of log-odds, Brier, MCC; per-class F1 with n and CI
rows = []
for key in BIN:
    ids, y, _, _ = ARR[key][MODELS[0]]
    idx, lab = IDX[key][:NB], LABEL[key]
    res = {m: binary_metrics(y, SC[key + (m,)].loc[ids, f"s_{lab}"].to_numpy(),
                             SC[key + (m,)].loc[ids, f"p_{lab}"].to_numpy(), ARR[key][m][2], idx) for m in MODELS}
    for metric in ("auroc", "auprc", "brier", "mcc"):
        sign = -1 if metric == "brier" else 1  # lower Brier is better
        best = max(MODELS, key=lambda m: sign * res[m][metric][0])
        mrows = []
        for m in MODELS:
            point, draws = res[m][metric]
            lo, hi = percentile_ci(draws, CONFIG["ci_level"])
            r = {"setting": TASK_NAME[key], "metric": metric, "model": m, "value": point, "ci_lo": lo, "ci_hi": hi,
                 "best": m == best}
            if m != best:
                r["p_boot"] = paired_bootstrap_p(draws - res[best][metric][1])
            mrows.append(r)
        others = [r for r in mrows if not r["best"]]
        for r, a in zip(others, holm([r["p_boot"] for r in others])):
            r["p_boot_holm"] = a
        rows += mrows
ph_save(rows, "threshold_free_binary", "PH", "AUROC and AUPRC of log-odds, Brier of P(Yes)/mass, MCC (strict), "
        "95% CI, paired bootstrap vs the best model by that metric, Holm over 14")

rows = []
for key in MC:
    ids, y, _, classes = ARR[key][MODELS[0]]
    n_gold = np.bincount(y, minlength=len(classes))
    for m in MODELS:
        point, draws = per_class_f1(y, ARR[key][m][2], len(classes), IDX[key][:NB_S])
        for c, cls in enumerate(classes):
            rows.append({"setting": TASK_NAME[key], "model": m, "class": cls, "n_gold": int(n_gold[c]), "f1": point[c],
                         "ci_lo": np.quantile(draws[:, c], 0.025), "ci_hi": np.quantile(draws[:, c], 0.975)})
ph_save(rows, "per_class_f1_ci", "PH", "EDOS B/C per-class F1 (strict) with gold support and 95% bootstrap CI (B=2,000)")

In [ ]:
# 2.3 Soft-label metrics against annotator shares (PH): L1 and Brier primary, cross-entropy in the appendix only
SOFT = {
    "cmsb": pd.read_parquet(DATA_ROOT / "processed" / "test" / "cmsb_test.parquet", columns=["id", "sexist_soft"]),
    "edos": pd.read_parquet(DATA_ROOT / "processed" / "test" / "edos_test.parquet",
                            columns=["id", "sexist_soft", "category_soft", "vector_soft"]),
}
SOFT = {ds: df.assign(id=lambda d: d["id"].astype(str)).set_index("id") for ds, df in SOFT.items()}


def soft_table(setting, preds, gold, refs):
    """L1/Brier (+CI) and appendix CE for every model and reference predictor on the same posts."""
    idx = bootstrap_indices(len(gold), NB, PH["seed"])
    out = []
    for name, q in {**preds, **refs}.items():
        losses = soft_losses(q, gold)
        r = {"setting": setting, "predictor": name, "reference": name in refs, "n": len(gold)}
        for metric in ("l1", "brier"):
            draws = losses[metric][idx].mean(1)
            r |= {metric: losses[metric].mean(), f"{metric}_lo": np.quantile(draws, 0.025),
                  f"{metric}_hi": np.quantile(draws, 0.975)}
        r["ce_appendix"] = losses["ce"].mean()
        out.append(r)
    return out


rows, excl = [], []
for key in BIN:
    ids, y, _, _ = ARR[key][MODELS[0]]
    gold = SOFT[key[0]].loc[ids, "sexist_soft"].to_numpy()
    preds = {m: SC[key + (m,)].loc[ids, f"p_{LABEL[key]}"].to_numpy() for m in MODELS}
    assert not np.isnan(np.stack(list(preds.values()))).any(), "zero-mass binary score"
    refs = {"ref: train prior": np.full(len(gold), PH["train_prior"][key[0]]), "ref: hard gold": y.astype(float),
            "ref: 15-model mean": np.mean(list(preds.values()), axis=0)}
    rows += soft_table(TASK_NAME[key], preds, gold, refs)
for key in MC:
    ids, y, _, classes = ARR[key][MODELS[0]]
    shares = np.array([[d[c] for c in classes] for d in SOFT["edos"].loc[ids, f"{get_schema(*key).column}_soft"]])
    q = {m: SC[key + (m,)].loc[ids, [f"q_{c}" for c in classes]].to_numpy() for m in MODELS}
    # Expert-override posts (no sexist annotator vote) have no category shares; zero-mass posts have no distribution
    keep = (shares.sum(1) > 0) & ~np.isnan(np.stack(list(q.values()))).any(axis=(0, 2))
    gold = shares[keep] / shares[keep].sum(1, keepdims=True)
    preds = {m: v[keep] for m, v in q.items()}
    refs = {"ref: train shares": np.tile(TRAIN_SHARES[key], (keep.sum(), 1)),
            "ref: hard gold": np.eye(len(classes))[y[keep]], "ref: 15-model mean": np.mean(list(preds.values()), axis=0)}
    rows += soft_table(TASK_NAME[key], preds, gold, refs)
    excl.append({"setting": TASK_NAME[key], "n_scored": len(ids), "n_no_sexist_votes": int((shares.sum(1) == 0).sum()),
                 "n_zero_mass": int((~keep).sum() - (shares.sum(1) == 0).sum())})
SOFT_TAB = ph_save(rows, "soft_metrics", "PH", "Mean L1 and Brier vs annotator shares (answer-forced predictions), "
                   "95% CI; cross-entropy (ε = 1e-6) for the appendix only; reference predictors marked")
ph_save(excl, "soft_metrics_exclusions", "PH", "EDOS B/C posts left out of the soft metrics and why")
tau_hs = []
for key in CONFIG["labelled"]:
    sub = SOFT_TAB[(SOFT_TAB.setting == TASK_NAME[key]) & ~SOFT_TAB.reference].set_index("predictor")
    t, p = kendalltau([MAIN_BY[key].loc[m, "macro_f1"] for m in MODELS], [-sub.loc[m, "l1"] for m in MODELS])
    tau_hs.append({"setting": TASK_NAME[key], "kendall_tau_macro_f1_vs_soft_l1": t, "p": p})
ph_save(tau_hs, "hard_vs_soft_ranking", "PH", "Kendall τ_b between the hard (macro-F1) and soft (L1) model rankings")

In [ ]:
# 2.4 Label-free decision rules (PH): train-prior-matched threshold, plain and prior-aware batch calibration
rows = []
for key in BIN + MC:
    ids, y, _, classes = ARR[key][MODELS[0]]
    idx, K = IDX[key][:NB], len(classes)
    tie_rank = np.argsort(np.argsort(np.array(ids)))  # deterministic tie-break by post id (docs/12 §2.4)
    for m in MODELS:
        sc = SC[key + (m,)].loc[ids]
        found = []
        if key in BIN:
            lab, prior = LABEL[key], PH["train_prior"][key[0]]
            s, p = sc[f"s_{lab}"].to_numpy(), sc[f"p_{lab}"].to_numpy()
            point, draws, cut = prior_matched(y, s, prior, idx, tie_rank)
            found.append(("prior-matched threshold", point, draws, round(prior * len(y)) / len(y), cut))
            q, cls_prior, answered = np.column_stack([1 - p, p]), np.array([1 - prior, prior]), sc[f"mass_{lab}"].to_numpy() >= T
        else:
            q = np.nan_to_num(sc[[f"q_{c}" for c in classes]].to_numpy(), nan=1 / K)  # zero mass = non-answer anyway
            cls_prior, answered = TRAIN_SHARES[key], sc["mass"].to_numpy() >= T
        for name, pr in (("batch calibration", None), ("prior-aware batch calibration", cls_prior)):
            point, draws, yp = batch_calibrated(y, q, idx, pr, answered)
            found.append((name, point, draws, float((yp == 1).mean()) if key in BIN else np.nan, np.nan))
        for name, point, draws, pos, cut in found:
            lo, hi = percentile_ci(draws, CONFIG["ci_level"])
            rows.append({"setting": TASK_NAME[key], "model": m, "rule": name, "macro_f1": point, "ci_lo": lo,
                         "ci_hi": hi, "primary_macro_f1": MAIN_BY[key].loc[m, "macro_f1"], "pred_pos_rate": pos,
                         "logodds_cut": cut})
ph_save(rows, "label_free_decision_rules", "PH", "Macro-F1 under label-free decision rules (prior from train), "
        "rules re-estimated in every resample; non-answers stay wrong; the prior-matched row measures ranking quality")

In [ ]:
# 2.5 Abstention views (PH): risk-coverage / AURC, and pairwise fair subsets vs the best model
rc, fair = [], []
for key in CONFIG["labelled"]:
    schema, (ids, y, _, classes) = get_schema(*key), ARR[key][MODELS[0]]
    K, best = len(classes), BEST[key]
    for m in MODELS:
        yp, sc = ARR[key][m][2], SC[key + (m,)].loc[ids]
        if key in BIN:
            conf = {"|log-odds|": np.abs(sc[f"s_{LABEL[key]}"].to_numpy())}
        else:
            conf = {"max answer-forced share": np.nan_to_num(sc[[f"q_{c}" for c in classes]].max(1).to_numpy()),
                    "answer mass": sc["mass"].to_numpy()}
        for cname, cv in conf.items():
            _, _, aurc, r80 = risk_coverage(yp == y, cv)
            rc.append({"setting": TASK_NAME[key], "model": m, "confidence": cname, "aurc": aurc, "risk_at_80pct": r80,
                       "risk_full_coverage": float((yp != y).mean())})
        if m == best:
            continue
        common = answered_ids(REC[key + (T,)][m], schema) & answered_ids(REC[key + (T,)][best], schema)
        mask = np.array([i in common for i in ids])
        r = {"setting": TASK_NAME[key], "model": m, "best": best, "n_common": int(mask.sum())}
        if mask.sum() > 1:
            idx2 = bootstrap_indices(int(mask.sum()), NB_S, PH["seed"])
            dm = bootstrap_metric(y[mask], yp[mask], K, idx2) - bootstrap_metric(y[mask], ARR[key][best][2][mask], K, idx2)
            fm, fb = macro_f1_counts(y[mask], yp[mask], K), macro_f1_counts(y[mask], ARR[key][best][2][mask], K)
            r |= {"macro_f1_model": fm, "macro_f1_best": fb, "delta": fm - fb, "delta_lo": np.quantile(dm, 0.025),
                  "delta_hi": np.quantile(dm, 0.975)}
        fair.append(r)
ph_save(rc, "risk_coverage", "PH", "AURC and selective risk at 80% coverage (|log-odds| for binary; answer-forced "
        "share and mass for option tasks)")
ph_save(fair, "pairwise_fair_subsets", "PH", "Each model vs the best on the posts both answered at mass ≥ 0.5, "
        "paired bootstrap 95% CI of Δ (B=2,000)")

In [ ]:
# 2.6 Failure analysis (PH): outcome and reason code per (model, post, label), plus the context that explains misses
LEX = load_lexicons()
raw_cmsb = (pd.read_csv(DATA_ROOT / "CMSB" / "sexism_data.csv", usecols=["id", "dataset", "toxicity"])
            .assign(id=lambda d: d["id"].astype(str)).set_index("id"))
CTX = {}
for ds in ("cmsb", "edos"):
    cols = ["id", "text_norm", "sexist", "sexist_soft"] + (["of_id"] if ds == "cmsb" else ["category", "vector"])
    t = pd.read_parquet(DATA_ROOT / "processed" / "test" / f"{ds}_test.parquet", columns=cols)
    t = t.assign(id=lambda d: d["id"].astype(str)).set_index("id")
    s = t["sexist_soft"]
    n_words = t["text_norm"].fillna("").str.split().str.len()
    ctx = pd.DataFrame({
        "support": s,
        "support_bin": np.select([s == 0, s < 0.5, s < 1], ["0", "(0,0.5)", "[0.5,1)"], "1"),
        "gender_term": word_hits(t["text_norm"], LEX["gender_terms"]),
        "gendered_slur": word_hits(t["text_norm"], LEX["gendered_slurs"]),
        "profanity": word_hits(t["text_norm"], LEX["profanity"]),
        "length": n_words.map(length_bin),
        # Gold disagrees with the annotator majority (EDOS gold is expert-adjudicated)
        "expert_override": ((t["sexist"] == 1) & (s < 0.5)) | ((t["sexist"] == 0) & (s > 0.5)),
    }, index=t.index)
    if ds == "cmsb":
        ctx["source"] = raw_cmsb.reindex(t.index)["dataset"]
        ctx["toxicity"] = raw_cmsb.reindex(t.index)["toxicity"]
        ctx["modification"] = t["of_id"].notna()
    else:
        ctx["category"], ctx["vector"] = t["category"], t["vector"]
        ctx["form"] = np.select([t["category"].isin(["derogation", "threats"]),
                                 t["category"].isin(["animosity", "prejudiced_discussions"])], ["explicit", "implicit"], None)
    CTX[ds] = ctx

FAIL = {}
(PH_DIR / "failure").mkdir(exist_ok=True)
for key in CONFIG["labelled"]:
    schema = get_schema(*key)
    gold = GOLD[key[0]].set_index("id")[schema.column]
    parts = []
    for m in MODELS:
        ft = failure_table(SC[key + (m,)], gold, schema, T).assign(model=m).join(CTX[key[0]], on="id")
        ft.to_parquet(PH_DIR / "failure" / f"{key[0]}_{key[1]}_{m}.parquet", index=False)  # ids and codes, no text
        parts.append(ft)
    FAIL[key] = pd.concat(parts, ignore_index=True).assign(setting=TASK_NAME[key])

binf = pd.concat([FAIL[k] for k in BIN], ignore_index=True)
pos = binf[binf.gold == True].assign(miss=lambda d: d.outcome != "TP")  # a non-answer on a sexist post is a miss
neg = binf[binf.gold == False].assign(false_flag=lambda d: d.outcome == "FP")
by_form = [rate_table(pos[pos.setting == "CMSB binary"], ["setting", "model", "source"], "miss", NB_S, PH["seed"])]
for col in ("category", "vector", "form", "gendered_slur", "length"):
    by_form.append(rate_table(pos[pos.setting == "EDOS A"], ["setting", "model", col], "miss", NB_S, PH["seed"]))
by_form.append(rate_table(pos[pos.setting == "CMSB binary"], ["setting", "model", "gendered_slur"], "miss", NB_S, PH["seed"]))
ph_save(pd.concat(by_form, ignore_index=True), "failure_fnr_by_form", "PH",
        "Miss rate (FN or non-answer) on gold-sexist posts by form of sexism: CMSB source, EDOS category/vector, "
        "explicit vs implicit, slur present, length; bootstrap 95% CI")
ph_save(pd.concat([rate_table(pos, ["setting", "model", "support_bin"], "miss", NB_S, PH["seed"]),
                   rate_table(neg, ["setting", "model", "support_bin"], "false_flag", NB_S, PH["seed"])], ignore_index=True),
        "failure_by_support", "PH", "Miss rate on sexist and false-flag rate on non-sexist posts by annotator support")
ph_save(rate_table(neg, ["setting", "model", "gender_term"], "false_flag", NB_S, PH["seed"]), "failure_fpr_gender_terms",
        "PH", "False-flag rate on non-sexist posts with vs without a gender term")
ph_save(pd.concat([rate_table(pos, ["setting", "model", "expert_override"], "miss", NB_S, PH["seed"]),
                   rate_table(neg, ["setting", "model", "expert_override"], "false_flag", NB_S, PH["seed"])], ignore_index=True),
        "failure_expert_override", "PH", "Miss and false-flag rates where the gold label overrides the annotator majority")
allf = pd.concat(FAIL.values(), ignore_index=True)
errs = allf[~allf.outcome.isin(["TP", "TN", "correct"])]
share = (errs.groupby(["setting", "model", "outcome", "reason"]).size()
         / errs.groupby(["setting", "model", "outcome"]).size()).rename("share").reset_index()
ph_save(share, "failure_reason_shares", "PH", "Reason codes among errors (FN, FP, wrong, non-answer) per model (docs/12 §2.6)")
na = allf[allf.outcome == "non_answer"]
ph_save((na.groupby(["setting", "model", "token_class"]).size() / na.groupby(["setting", "model"]).size()).rename("share")
        .reset_index(), "failure_nonanswer_tokens", "PH", "First-token class of non-answers (format vs refusal tokens)")

fnr_vec = pos[pos.setting == "EDOS A"].groupby(["model", "vector"]).miss.mean().unstack().loc[MODELS]
fig, ax = plt.subplots(figsize=(7.5, 4.6))
im = heatmap(ax, fnr_vec.rename(index=DISPLAY), "EDOS A test: miss rate on sexist posts by gold vector (PH)")
fig.colorbar(im, ax=ax, shrink=0.7)
fig.savefig(PH_FIGS / "failure_fnr_by_vector.pdf")
plt.show()

In [ ]:
# 2.7 Label-free reference baselines and 2.8 CMSB contrast set (PH)
rows = []
for key in BIN:
    ids, y, _, _ = ARR[key][MODELS[0]]
    idx, ctx = IDX[key][:NB], CTX[key[0]].loc[ids]
    rules = {"gender-term rule": ctx["gender_term"].to_numpy().astype(int)}
    if key[0] == "cmsb":
        rules["Perspective toxicity >= 0.5"] = (ctx["toxicity"].to_numpy() >= 0.5).astype(int)
    for name, yp in rules.items():
        lo, hi = percentile_ci(bootstrap_metric(y, yp, 2, idx), CONFIG["ci_level"])
        rows.append({"setting": TASK_NAME[key], "baseline": name, "macro_f1": macro_f1_counts(y, yp, 2), "ci_lo": lo,
                     "ci_hi": hi, "pred_pos_rate": float(yp.mean())})
    if key[0] == "cmsb":
        tox = ctx["toxicity"].to_numpy()
        auroc, draws = binary_metrics(y, tox, tox, (tox >= 0.5).astype(int), idx)["auroc"]
        rows.append({"setting": TASK_NAME[key], "baseline": "Perspective toxicity (AUROC)", "auroc": auroc,
                     "ci_lo": np.quantile(draws, 0.025), "ci_hi": np.quantile(draws, 0.975)})
ph_save(rows, "reference_baselines", "PH", "Label-free reference rules scored like a model (strict macro-F1, 95% CI)")

key = ("cmsb", "binary")
ids, y, _, _ = ARR[key][MODELS[0]]
meta = (pd.read_parquet(DATA_ROOT / "processed" / "test" / "cmsb_test.parquet", columns=["id", "of_id"])
        .assign(id=lambda d: d["id"].astype(str)))
PAIRS = contrast_pairs(meta[meta.id.isin(ids)])
gold_s = pd.Series(y.astype(bool), index=ids)
src = CTX["cmsb"]["source"]
mods, origs = set(PAIRS["mod"]), set(PAIRS["orig"])
sources = sorted(src.loc[sorted(origs)].dropna().unique())
neg_mod = [i for i in ids if not gold_s[i] and i in mods]
neg_other = [i for i in ids if not gold_s[i] and i not in mods]
pos_orig = [i for i in sorted(origs) if gold_s[i]]
rows = []
for m in MODELS:
    pred = pd.Series(ARR[key][m][2], index=ids)
    s = SC[key + (m,)].loc[ids, f"s_{LABEL[key]}"]
    rows.append({"model": m, "source": "all", **contrast_metrics(PAIRS, gold_s, pred, s, NB_S, PH["seed"]),
                 "tnr_modifications": float((pred.loc[neg_mod] == 0).mean()),
                 "tnr_other_non_sexist": float((pred.loc[neg_other] == 0).mean()),
                 "recall_originals_with_mods": float((pred.loc[pos_orig] == 1).mean())})
    for source in sources:
        pairs = PAIRS[src.loc[PAIRS["orig"]].to_numpy() == source]
        rows.append({"model": m, "source": source, **contrast_metrics(pairs, gold_s, pred, s, NB_S, PH["seed"])})
ph_save(rows, "cmsb_contrast_set", "PH", "CMSB original/adversarial-modification pairs: pair accuracy, within-pair "
        "log-odds gap (pair bootstrap 95% CI), TNR on modifications vs other non-sexist posts (docs/12 §2.8)")

In [ ]:
# 3.8 Explanation sample (PR): per (model, task) pair, seeded posts per TP/FP/FN/TN cell; notebook 40 gets ids only
gemma = "gemma4-e4b"
XAI_PAIRS = [(("cmsb", "binary"), BEST[("cmsb", "binary")]), (("edos", "task_a"), BEST[("edos", "task_a")])]
if gemma != BEST[("cmsb", "binary")]:
    XAI_PAIRS.append((("cmsb", "binary"), gemma))
XAI_DIR = PH_DIR / "xai_ids"
XAI_DIR.mkdir(exist_ok=True)


def xai_sample(ft, key, m, split):
    """Write the shuffled id list (for notebook 40) and the FN/FP list (self-explanations); cells stay here."""
    rng = np.random.default_rng(PH["seed"])
    cells = []
    for cell in ("TP", "FP", "FN", "TN"):
        ids_c = sorted(ft.loc[ft.outcome == cell, "id"])
        pick = rng.choice(ids_c, size=min(PH["xai_per_cell"], len(ids_c)), replace=False) if ids_c else []
        cells += [{"id": i, "cell": cell} for i in pick]
    cells = pd.DataFrame(cells)
    stem = f"{key[0]}-{key[1]}-{m}-{split}"
    cells.to_csv(XAI_DIR / f"{stem}_cells.csv", index=False)  # outcome per id: stays in 99's run folder
    shuffled = rng.permutation(cells["id"].to_numpy()) if len(cells) else []
    (XAI_DIR / f"{stem}.txt").write_text("".join(f"{i}\n" for i in shuffled))
    fnfp = sorted(cells.loc[cells.cell.isin(["FN", "FP"]), "id"]) if len(cells) else []
    (XAI_DIR / f"{stem}-fnfp.txt").write_text("".join(f"{i}\n" for i in rng.permutation(fnfp)))
    return {"pair": stem, **cells.cell.value_counts().to_dict()} if len(cells) else {"pair": stem}


xai_rows = [xai_sample(FAIL[key][FAIL[key].model == m], key, m, "test") for key, m in XAI_PAIRS]
ph_save(xai_rows, "xai_sample_sizes", "PR", "Explanation sample per (task, model): posts per outcome cell (docs/12 §3.8)")
print("id lists for notebook 40:", sorted(p.name for p in XAI_DIR.glob("*.txt")))

In [ ]:
# 3.1-3.2 Dev sweep (PR): dev-tuned rules applied to test, EXIST English dev evaluation, EXIST dev failure analysis
DEV = exp_runs("devsweep", "dev")
if DEV is None:
    print("dev sweep not run yet (PH['experiments']['devsweep'] is None): skipped")
else:
    DEV_RUNS = {(r["dataset"], r["task"], r["model"]): r for r in DEV}
    DEV_GOLD = {ds: pd.read_parquet(DATA_ROOT / "processed" / "dev" / f"{ds}_dev.parquet")
                .assign(id=lambda d: d["id"].astype(str)) for ds in ("cmsb", "edos", "exist")}
    dev_rows = []
    for key in BIN:
        ids, y, _, _ = ARR[key][MODELS[0]]
        lab, idx = LABEL[key], IDX[key][:NB]
        for m in MODELS:
            if key + (m,) not in DEV_RUNS:
                continue
            dsc = label_scores(DEV_RUNS[key + (m,)]["records"], get_schema(*key))
            g = DEV_GOLD[key[0]].set_index("id").loc[dsc.index, "sexist"].astype(int).to_numpy()
            s_dev, s_te = dsc[f"s_{lab}"].to_numpy(), SC[key + (m,)].loc[ids, f"s_{lab}"].to_numpy()
            answered = SC[key + (m,)].loc[ids, f"mass_{lab}"].to_numpy() >= T
            t_star, temp, (a, b) = fit_threshold(s_dev, g), fit_temperature(s_dev, g), fit_platt(s_dev, g)
            for rule, yp in (("dev-tuned threshold", (s_te >= t_star).astype(int)),
                             ("dev Platt scaling", (a * s_te + b >= 0).astype(int))):
                yp = np.where(answered, yp, 1 - y)  # strict: a non-answer is wrong
                lo, hi = percentile_ci(bootstrap_metric(y, yp, 2, idx), CONFIG["ci_level"])
                dev_rows.append({"setting": TASK_NAME[key], "model": m, "rule": rule, "macro_f1": macro_f1_counts(y, yp, 2),
                                 "ci_lo": lo, "ci_hi": hi, "threshold": t_star if "threshold" in rule else np.nan,
                                 "pred_pos_rate": float(yp.mean()), "primary_macro_f1": MAIN_BY[key].loc[m, "macro_f1"]})
            p_t = 1 / (1 + np.exp(-s_te / temp))
            dev_rows.append({"setting": TASK_NAME[key], "model": m, "rule": "dev temperature (calibration only)",
                             "temperature": temp, "brier": float(np.mean((p_t - y) ** 2)),
                             "ece": expected_calibration_error(y, p_t, CONFIG["ece_bins"])[0]})
    for key in MC:
        ids, y, _, classes = ARR[key][MODELS[0]]
        schema, idx = get_schema(*key), IDX[key][:NB]
        for m in MODELS:
            if key + (m,) not in DEV_RUNS:
                continue
            drecs = DEV_RUNS[key + (m,)]["records"]
            dgold = DEV_GOLD[key[0]].assign(id=lambda d: d["id"].astype(str))
            grid = [round(0.1 * k, 1) for k in range(1, 10)]
            f = [macro_f1_counts(*strict_arrays(dgold, apply_mass_threshold(drecs, tau), schema)[1:3], len(classes)) for tau in grid]
            tau = max(t for t, v in zip(grid, f) if v >= max(f) - 1e-12)  # ties -> larger tau (docs/12 §3.1b)
            yp = strict_arrays(GOLD[key[0]], apply_mass_threshold(RUNS[key + (m,)]["records"], tau), schema)[2]
            lo, hi = percentile_ci(bootstrap_metric(y, yp, len(classes), idx), CONFIG["ci_level"])
            dev_rows.append({"setting": TASK_NAME[key], "model": m, "rule": "dev-chosen non-answer cutoff", "tau": tau,
                             "macro_f1": macro_f1_counts(y, yp, len(classes)), "ci_lo": lo, "ci_hi": hi,
                             "primary_macro_f1": MAIN_BY[key].loc[m, "macro_f1"]})
    ph_save(dev_rows, "dev_tuned_rules_on_test", "PR", "Rules fitted on dev (threshold, Platt, temperature, non-answer "
            "cutoff) applied to the test predictions (docs/12 §3.1)")

    # EXIST English dev with the official metrics (PyEvALL) and our strict macro-F1 (docs/12 §3.2)
    en = set(DEV_GOLD["exist"].query("lang == 'en'")["id"])
    GOLD_DIR, BASE_DIR = DATA_ROOT / "EXIST" / "evaluation" / "golds", DATA_ROOT / "EXIST" / "evaluation" / "baselines"
    xrows = []
    for task in ("task_1_1", "task_1_2", "task_1_3"):
        schema, ev = get_schema("exist", task), task.replace("task_", "task")
        for soft in (False, True):
            gold = gold_subset(GOLD_DIR / f"EXIST2025_dev_{ev}_gold_{'soft' if soft else 'hard'}.json", en)
            metrics = ["ICMSoft", "ICMSoftNorm", "CrossEntropy"] if soft else ["ICM", "ICMNorm", "FMeasure"]
            base = json.loads((BASE_DIR / f"EXIST2025_dev_{ev}_majority_class_{'soft' if soft else 'hard'}.json").read_text())
            xrows.append({"task": task, "type": "soft" if soft else "hard", "model": "majority-class baseline",
                          "variant": "official", "n_items": len([b for b in base if b["id"] in en]),
                          **pyevall_scores([b for b in base if b["id"] in en], gold, metrics, task)})
            for m in MODELS:
                if ("exist", task, m) not in DEV_RUNS:
                    continue
                for variant, t in (("omit", T), ("answer-forced", None)):
                    recs = apply_mass_threshold(DEV_RUNS[("exist", task, m)]["records"], t)
                    parents = (apply_mass_threshold(DEV_RUNS[("exist", "task_1_1", m)]["records"], t)
                               if task == "task_1_3" and ("exist", "task_1_1", m) in DEV_RUNS else None)
                    items = [it for it in evall_items(recs, schema, soft, parents) if it["id"] in en]
                    xrows.append({"task": task, "type": "soft" if soft else "hard", "model": m, "variant": variant,
                                  "n_items": len(items), "n_gold": len(gold), **pyevall_scores(items, gold, metrics, task)})
    ph_save(xrows, "exist_dev_official_metrics", "PR", "EXIST 2025 English dev (n = 444/400/444 hard, 489 soft): "
            "ICM, ICM-Norm, F1 / ICM-Soft(-Norm), CE via PyEvALL 0.2.11; omit = non-answers left out (lenient)")
    srows = []
    dg = DEV_GOLD["exist"].query("lang == 'en'")
    for task in ("task_1_1", "task_1_2", "task_1_3"):
        schema = get_schema("exist", task)
        for m in MODELS:
            if ("exist", task, m) not in DEV_RUNS:
                continue
            recs = apply_mass_threshold(DEV_RUNS[("exist", task, m)]["records"], T)
            if schema.task_type == "multilabel":
                g = dg.set_index("id")["labels"].dropna()
                pr = {r["id"]: {s["label"]: (s["pred"] if not s["refused"] else None) for s in r["scores"]} for r in recs}
                ids_ = [i for i in g.index if i in pr]
                Y = np.array([[lab in g[i] for lab in schema.labels()] for i in ids_])
                YP = np.array([[(not Y[j, c]) if pr[i][lab] is None else bool(pr[i][lab])
                                for c, lab in enumerate(schema.labels())] for j, i in enumerate(ids_)])
                point, draws = multilabel_macro_f1(Y, YP, bootstrap_indices(len(ids_), NB_S, PH["seed"]))
                srows.append({"task": task, "model": m, "n": len(ids_), "macro_f1": point,
                              "ci_lo": np.quantile(draws, 0.025), "ci_hi": np.quantile(draws, 0.975)})
                continue
            ids_, y, yp, classes = strict_arrays(dg, recs, schema)
            idx2 = bootstrap_indices(len(ids_), NB_S, PH["seed"])
            draws = bootstrap_metric(y, yp, len(classes), idx2)
            r = {"task": task, "model": m, "n": len(ids_), "macro_f1": macro_f1_counts(y, yp, len(classes)),
                 "ci_lo": np.quantile(draws, 0.025), "ci_hi": np.quantile(draws, 0.975)}
            if schema.task_type == "binary":
                r["f1_positive"] = f1_score(y, yp, zero_division=0)
                r["pred_pos_rate"], r["gold_pos_rate"] = float((yp == 1).mean()), float(y.mean())
            srows.append(r)
    EXIST_DEV = ph_save(srows, "exist_dev_strict", "PR", "EXIST English dev strict macro-F1 (and positive-class F1, "
                        "predicted vs gold positive rate for 1.1), bootstrap 95% CI (B=2,000); LLM-only, no ranking claims")

    # EXIST dev failure analysis: which intentions and categories are missed (docs/12 §2.6 on dev)
    dctx = dg.set_index("id")
    schema = get_schema("exist", "task_1_1")
    fparts = []
    for m in MODELS:
        if ("exist", "task_1_1", m) not in DEV_RUNS:
            continue
        ft = failure_table(label_scores(DEV_RUNS[("exist", "task_1_1", m)]["records"], schema),
                           dctx["sexist"].dropna().astype(bool), schema, T).assign(model=m)
        ft = ft.join(dctx[["intention", "sexist_soft"]], on="id")
        ft["gender_term"] = word_hits(dctx.loc[ft["id"], "text_norm"].reset_index(drop=True), LEX["gender_terms"]).to_numpy()
        fparts.append(ft)
    if fparts:
        xf = pd.concat(fparts, ignore_index=True)
        xpos = xf[xf.gold == True].assign(miss=lambda d: d.outcome != "TP")
        ph_save(rate_table(xpos, ["model", "intention"], "miss", NB_S, PH["seed"]), "exist_dev_fnr_by_intention", "PR",
                "EXIST 1.1 dev miss rate on sexist tweets by source intention (DIRECT / REPORTED / JUDGEMENTAL)")
        best_x = EXIST_DEV[EXIST_DEV.task == "task_1_1"].sort_values("f1_positive", ascending=False).model.tolist()
        taken = {m for _, m in XAI_PAIRS}
        pick = next((m for m in best_x if m not in taken), best_x[0])
        for m in dict.fromkeys([pick, gemma]):
            xai_rows.append(xai_sample(xf[xf.model == m], ("exist", "task_1_1"), m, "dev"))
        ph_save(xai_rows, "xai_sample_sizes", "PR", "Explanation sample per (task, model, split): posts per outcome cell")

In [ ]:
# 3.3 Format check (PR, label-free) and 3.4 answer prefix v2 (PR)
FC = exp_runs("format_check", "test")
if FC is None:
    print("format check not run yet: skipped")
else:
    rows = []
    for r in FC:
        key, m = (r["dataset"], r["task"]), r["model"]
        frozen = {x["id"]: x["scores"][0] for x in RUNS[key + (m,)]["records"]}
        recs = [x for x in r["records"] if x["id"] in frozen]
        parsed = [x["parsed"][get_schema(*key).column] if x["parsed"] else None for x in recs]
        first = [frozen[x["id"]]["pred"] for x in recs]
        non_ans = [frozen[x["id"]]["mass"] < T for x in recs]
        both = [(p, f) for p, f, na in zip(parsed, first, non_ans) if p is not None and not na]
        rows.append({"setting": TASK_NAME[key], "model": m, "n": len(recs),
                     "unparseable_rate": float(np.mean([p is None for p in parsed])),
                     "mismatch_rate_answered": float(np.mean([p != f for p, f in both])) if both else np.nan,
                     "parsed_among_first_token_non_answers": float(np.mean([p is not None for p, na in zip(parsed, non_ans) if na]))
                     if any(non_ans) else np.nan,
                     **{f"rule_{k}": v for k, v in pd.Series([x["parse_rule"] for x in recs]).value_counts(normalize=True).items()}})
    ph_save(rows, "format_check", "PR", "Free-text reply vs first-token answer on a seeded test subset (label-free)")

AP = exp_runs("answer_prefix", "test")
if AP is None:
    print("answer-prefix runs not run yet: skipped")
else:
    rows = []
    for r in AP:
        key, m = (r["dataset"], r["task"]), r["model"]
        schema = get_schema(*key)
        recs = apply_mass_threshold(r["records"], T)
        row = {"setting": TASK_NAME[key], "model": m, "variant": r.get("prompt_variant"),
               "non_answer_rate_v2": 1 - len(answered_ids(recs, schema)) / len(recs),
               "non_answer_rate_v1": 1 - len(answered_ids(REC[key + (T,)][m], schema)) / len(REC[key + (T,)][m])}
        if key in MC:
            ids, y, yp, classes = strict_arrays(GOLD[key[0]], recs, schema)
            idx2 = bootstrap_indices(len(ids), NB, PH["seed"])
            lo, hi = percentile_ci(bootstrap_metric(y, yp, len(classes), idx2), CONFIG["ci_level"])
            row |= {"n": len(ids), "macro_f1_v2": macro_f1_counts(y, yp, len(classes)), "ci_lo": lo, "ci_hi": hi,
                    "macro_f1_v1": MAIN_BY[key].loc[m, "macro_f1"]}
        rows.append(row)
    ph_save(rows, "answer_prefix_v2", "PR", "v2 'Answer: ' prefix vs v1: non-answer rate and strict macro-F1 (docs/12 §3.4)")

In [ ]:
# 3.5 Option-order control (PR): permutation consistency, letter choice, per-order macro-F1, PriDe on EDOS C
OO = exp_runs("option_order", "test")
if OO is None:
    print("option-order runs not run yet: skipped")
else:
    cons, letters, perf = [], [], []
    by = {}
    for r in OO:
        by.setdefault((r["dataset"], r["task"], r["model"]), []).append(r)
    for (ds, task, m), runs in by.items():
        key, schema = (ds, task), get_schema(ds, task)
        tmpl = PromptTemplate.load(schema)
        full = [r for r in runs if not r.get("subset")]
        if full:  # cyclic shifts on all posts (EDOS B, EXIST 1.2) next to the frozen order 0
            orders = {0: {x["id"]: x["scores"][0]["pred"] for x in RUNS[key + (m,)]["records"]}}
            orders |= {r["option_perm"]: {x["id"]: x["scores"][0]["pred"] for x in r["records"]} for r in full}
            common = set.intersection(*(set(o) for o in orders.values()))
            same = [len({orders[k][i] for k in orders}) == 1 for i in common]
            cons.append({"setting": TASK_NAME[key], "model": m, "n_orders": len(orders), "n": len(common),
                         "consistency": float(np.mean(same))})
            for k, preds in orders.items():
                order = tmpl.option_order(k)
                pos_ = pd.Series([order.index(preds[i]) for i in common if preds[i] is not None])
                letters.append({"setting": TASK_NAME[key], "model": m, "perm": k,
                                **{f"letter_{chr(65 + j)}": v for j, v in pos_.value_counts(normalize=True).sort_index().items()}})
            if key in MC:
                for r in full:
                    ids, y, yp, classes = strict_arrays(GOLD[ds], apply_mass_threshold(r["records"], T), schema)
                    perf.append({"setting": TASK_NAME[key], "model": m, "perm": r["option_perm"], "n": len(ids),
                                 "macro_f1": macro_f1_counts(y, yp, len(classes))})
        sub = sorted((r for r in runs if r.get("subset")), key=lambda r: r.get("option_perm") or 0)
        if sub and key in MC:  # PriDe: letter prior from the subset under all cyclic orders, then debias the rest
            classes, K = schema.labels(), len(schema.labels())
            ids_p = sorted(set.intersection(*({x["id"] for x in r["records"]} for r in sub)))
            logp = np.full((len(ids_p), K, K), np.log(1 / K))
            for r in sub:
                k, order, recd = r.get("option_perm") or 0, tmpl.option_order(r.get("option_perm") or 0), {x["id"]: x for x in r["records"]}
                for a, i in enumerate(ids_p):
                    sc = recd[i]["scores"][0]
                    q = np.array([sc["probs"][c] for c in order]) / max(sc["mass"], 1e-12)
                    logp[a, k] = np.log(np.maximum(q, 1e-12))
            prior = pride_prior(logp)
            ids, y, yp0, _ = strict_arrays(GOLD[ds], REC[key + (T,)][m], schema)
            keep = np.array([i not in set(ids_p) for i in ids])
            q0 = np.nan_to_num(SC[key + (m,)].loc[ids, [f"q_{c}" for c in classes]].to_numpy(), nan=1 / K)
            yp = np.where(SC[key + (m,)].loc[ids, "mass"].to_numpy() >= T, pride_debias(q0, prior).argmax(1), K)
            perf.append({"setting": TASK_NAME[key], "model": m, "perm": "PriDe-debiased", "n": int(keep.sum()),
                         "macro_f1": macro_f1_counts(y[keep], yp[keep], K),
                         "macro_f1_frozen_same_posts": macro_f1_counts(y[keep], yp0[keep], K),
                         **{f"prior_{chr(65 + j)}": v for j, v in enumerate(prior)}})
    ph_save(cons, "option_order_consistency", "PR", "Share of posts with the same predicted class under all cyclic option orders (label-free)")
    ph_save(letters, "option_order_letters", "PR", "Share of predictions per answer letter, per option order (label-free)")
    ph_save(perf, "option_order_macro_f1", "PH", "Strict macro-F1 per option order, and PriDe-debiased EDOS C (docs/12 §3.5)")

In [ ]:
# 3.6 Five-prompt study PV-1 (PR): macro-F1 per template, spread, Kendall's W, variance shares, two-way bootstrap
P5 = exp_runs("prompt5", "test")
if P5 is None:
    print("prompt study not run yet: skipped")
else:
    from src.utils import kendalls_w, pigeonhole_bootstrap, variance_shares

    runs5 = {(r["dataset"], r["task"], r["model"], r.get("prompt_variant")): r for r in P5}
    per, summ, agree = [], [], []
    for key in CONFIG["labelled"]:
        schema = get_schema(*key)
        tmpls = ["T0"] + sorted({v for (ds, t, _, v) in runs5 if (ds, t) == key})
        models = [m for m in MODELS if all(v == "T0" or key + (m, v) in runs5 for v in tmpls)]
        if len(tmpls) < 2 or not models:
            continue
        arrs = {}
        for m in models:
            for v in tmpls:
                recs = REC[key + (T,)][m] if v == "T0" else apply_mass_threshold(runs5[key + (m, v)]["records"], T)
                arrs[(m, v)] = strict_arrays(GOLD[key[0]], recs, schema)
        common = sorted(set.intersection(*(set(a[0]) for a in arrs.values())))
        pos_of = {v: {i: j for j, i in enumerate(a[0])} for v, a in arrs.items()}
        y = np.array([arrs[(models[0], "T0")][1][pos_of[(models[0], "T0")][i]] for i in common])
        K = len(arrs[(models[0], "T0")][3])
        mat = np.zeros((len(models), len(tmpls)))
        for a, m in enumerate(models):
            yp_t = np.stack([[arrs[(m, v)][2][pos_of[(m, v)][i]] for i in common] for v in tmpls])
            for b, v in enumerate(tmpls):
                mat[a, b] = macro_f1_counts(y, yp_t[b], K)
                per.append({"setting": TASK_NAME[key], "model": m, "template": v, "n": len(common), "macro_f1": mat[a, b]})
            draws = pigeonhole_bootstrap(y, yp_t, K, NB_S, PH["seed"])
            summ.append({"setting": TASK_NAME[key], "model": m, "mean": mat[a].mean(), "sd": mat[a].std(ddof=1),
                         "min": mat[a].min(), "max": mat[a].max(), "ci_lo": np.quantile(draws, 0.025),
                         "ci_hi": np.quantile(draws, 0.975)})
        ranks = pd.DataFrame(mat, index=models, columns=tmpls).rank(ascending=False)
        for m in models:
            summ[-len(models) + models.index(m)] |= {"rank_min": ranks.loc[m].min(), "rank_max": ranks.loc[m].max()}
        agree.append({"setting": TASK_NAME[key], "n_templates": len(tmpls), "kendalls_w": kendalls_w(mat),
                      "top_model_share": float((mat.argmax(0) == mat.mean(1).argmax()).mean()), **variance_shares(mat)})
    ph_save(per, "prompt5_per_template", "PR", "Strict macro-F1 per model and template (T0 = frozen v1)")
    ph_save(summ, "prompt5_summary", "PR", "Mean ± SD over templates, min/max, rank range, two-way bootstrap 95% CI")
    ph_save(agree, "prompt5_agreement", "PR", "Kendall's W across templates, share of templates where the top model "
            "stays top, model/template/residual shares of variance")

In [ ]:
# 3.7 Encoder baselines (PR): strict macro-F1 per seed with CI, mean ± std over seeds, dev-tuned threshold row
ENC = exp_runs("encoders", "test", kind="encoder")
if ENC is None:
    print("encoder runs not run yet: skipped")
else:
    from src.model.encoder import encoder_arrays

    rows = []
    for r in ENC:
        key = (r["dataset"], r["task"])
        if key not in CONFIG["labelled"]:
            continue
        ids, y, _, classes = ARR[key][MODELS[0]]
        have = {x["id"] for x in r["records"]}
        keep = np.array([i in have for i in ids])
        # A real encoder run predicts every scored test post; only sanity runs cover a few
        assert SAN or keep.all(), f"{r['run_name']} lacks {int((~keep).sum())} scored posts"
        ids, y = [i for i, k in zip(ids, keep) if k], y[keep]
        yp, probs = encoder_arrays(r["records"], ids, classes)
        idx_e = IDX[key][:NB] if keep.all() else bootstrap_indices(len(ids), NB, PH["seed"])
        lo, hi = percentile_ci(bootstrap_metric(y, yp, len(classes), idx_e), CONFIG["ci_level"])
        row = {"setting": TASK_NAME[key], "model": r["model_name"], "seed": r["seed"], "macro_f1": macro_f1_counts(y, yp, len(classes)),
               "ci_lo": lo, "ci_hi": hi, "best_epoch": json.loads((r["run_dir"] / "dev_metrics.json").read_text())["best_epoch"]}
        if key in BIN:  # dev-tuned threshold on the encoder's own dev log-odds
            dev = [json.loads(line) for line in (r["run_dir"] / "dev_predictions.jsonl").read_text().splitlines()]
            dg = DEV_GOLD[key[0]] if "DEV_GOLD" in globals() else pd.read_parquet(
                DATA_ROOT / "processed" / "dev" / f"{key[0]}_dev.parquet").assign(id=lambda d: d["id"].astype(str))
            g = dg.set_index("id").loc[[d["id"] for d in dev], "sexist"].astype(int).to_numpy()
            s_dev = np.array([np.log(max(d["probs"]["True"], 1e-12)) - np.log(max(d["probs"]["False"], 1e-12)) for d in dev])
            s_te = np.log(np.maximum(probs[:, 1], 1e-12)) - np.log(np.maximum(probs[:, 0], 1e-12))
            row["macro_f1_dev_threshold"] = macro_f1_counts(y, (s_te >= fit_threshold(s_dev, g)).astype(int), 2)
        rows.append(row)
    ENC_TAB = ph_save(rows, "encoders_per_seed", "PR", "Fine-tuned encoders: strict test macro-F1 per seed with 95% CI")
    ph_save(ENC_TAB.groupby(["setting", "model"]).macro_f1.agg(["mean", "std", "min", "max", "count"]).reset_index(),
            "encoders_summary", "PR", "Fine-tuned encoders: mean ± std, min-max over seeds (reference point, no claims)")

In [ ]:
# 3.8 Explainability joins (PR): faithfulness by outcome, SHAP-LIME agreement, lexicon effects, plausibility,
# counterfactual flip rates by gold class, self-explanation codes; outcome cells come from the id lists above
XAI = None if PH["experiments"]["xai"] is None else exp_runs("xai", "test", "xai") + exp_runs("xai", "dev", "xai")
if XAI is None:
    print("explainability runs not run yet: skipped")
else:
    import difflib

    faith, lexr, cf, plaus = [], [], [], []
    for r in XAI:
        key, m, rd = (r["dataset"], r["task"]), r["model_name"], r["run_dir"]
        cells_path = XAI_DIR / f"{key[0]}-{key[1]}-{m}-{r['split']}_cells.csv"
        cells = pd.read_csv(cells_path, dtype=str).set_index("id")["cell"] if cells_path.exists() else pd.Series(dtype=str)
        exps = [json.loads(line) for line in (rd / "explanations.jsonl").read_text().splitlines()] if (rd / "explanations.jsonl").exists() else []
        for e in exps:
            row = {"setting": TASK_NAME[key], "model": m, "id": e["id"], "cell": cells.get(e["id"])}
            row |= {k: v for k, v in e["faithfulness"].items()} | {"shap_lime_rho": e.get("shap_lime_rho"),
                    "shap_lime_top3_overlap": e.get("shap_lime_top3_overlap"), "lime_seed_rho": e.get("lime_seed_rho")}
            faith.append(row)
            for tok, val, lc in zip(e["shap"]["tokens"], e["shap"]["values"], e["shap"]["lexicon_class"]):
                lexr.append({"setting": TASK_NAME[key], "model": m, "lexicon_class": lc, "shap": val})
        if key == ("cmsb", "binary") and exps and len(PAIRS):
            text = pd.read_parquet(DATA_ROOT / "processed" / "test" / "cmsb_test.parquet", columns=["id", "text_norm"]) \
                .assign(id=lambda d: d["id"].astype(str)).set_index("id")["text_norm"]
            first_mod = PAIRS.groupby("orig")["mod"].first()
            for e in exps:
                if e["id"] not in first_mod.index or cells.get(e["id"]) != "TP":
                    continue
                a, b = text[e["id"]].lower().split(), text[first_mod[e["id"]]].lower().split()
                changed = {w for op, i1, i2, _, _ in difflib.SequenceMatcher(a=a, b=b).get_opcodes() if op != "equal" for w in a[i1:i2]}
                ranked = [t.strip().lower().strip(".,!?;:\"'") for t, _ in sorted(zip(e["shap"]["tokens"], e["shap"]["values"]), key=lambda x: -x[1])]
                k = max(len(changed), 1)
                plaus.append({"model": m, "id": e["id"], "n_changed": len(changed),
                              "precision_at_k": float(np.mean([w in changed for w in ranked[:k]]))})
        if (rd / "counterfactual.jsonl").exists():
            cfd = pd.DataFrame([json.loads(line) for line in (rd / "counterfactual.jsonl").read_text().splitlines()])
            gold = (GOLD[key[0]].set_index("id")["sexist"] if r["split"] == "test" else
                    DEV_GOLD[key[0]].set_index("id")["sexist"] if "DEV_GOLD" in globals() else None)
            if gold is not None:
                cfd["gold"] = gold.reindex(cfd["id"]).to_numpy()
                for test_name, col in (("gender swap", "s_swap"), ("slurs/profanity removed", "s_noslur")):
                    changed = cfd[cfd[f"{col}_changed"]]
                    for gcls, grp in changed.groupby("gold"):
                        flips = (grp["s_orig"] >= 0) != (grp[col] >= 0)
                        cf.append({"setting": TASK_NAME[key], "model": m, "split": r["split"], "test": test_name,
                                   "gold_sexist": bool(gcls), "n_changed": len(grp), "flip_rate": float(flips.mean()),
                                   "mean_delta_logodds": float((grp[col] - grp["s_orig"]).mean())})
    if faith:
        ph_save(pd.DataFrame(faith).groupby(["setting", "model", "cell"]).mean(numeric_only=True).reset_index(),
                "xai_faithfulness_by_cell", "PR", "ERASER comprehensiveness/sufficiency of top SHAP words vs random words, "
                "mask and delete, and SHAP-LIME agreement, by outcome cell (docs/12 §3.8)")
    if lexr:
        ph_save(pd.DataFrame(lexr).groupby(["setting", "model", "lexicon_class"]).shap.agg(["mean", "count"]).reset_index(),
                "xai_lexicon_effects", "PR", "Mean SHAP value (log-odds toward Yes) by word class")
    if plaus:
        ph_save(pd.DataFrame(plaus).groupby("model").precision_at_k.agg(["mean", "count"]).reset_index(),
                "xai_plausibility_cmsb", "PR", "Precision@k of top SHAP words against the words the CMSB modification changed")
    if cf:
        ph_save(cf, "xai_counterfactual_flips", "PR", "Decision flip rate and mean Δ log-odds after a gender swap or "
                "after removing slurs/profanity, by gold class")
    # Self-explanations: model-claimed reasons for FN/FP, coded into the fixed taxonomy (claims, not faithful explanations)
    codes = []
    for r in exp_runs("xai", "test", "xai-codes") + exp_runs("xai", "dev", "xai-codes"):
        key, m = (r["dataset"], r["task"]), r["model_name"]
        cells_path = XAI_DIR / f"{key[0]}-{key[1]}-{m}-{r['split']}_cells.csv"
        cells = pd.read_csv(cells_path, dtype=str).set_index("id")["cell"] if cells_path.exists() else pd.Series(dtype=str)
        for c in r["records"]:
            codes.append({"setting": TASK_NAME[key], "model": m, "id": c["id"], "cell": cells.get(c["id"]),
                          "code": c["code"], "parsed_answer": c.get("parsed_answer")})
        filled = r["run_dir"] / "human_coding_filled.csv"
        if filled.exists():  # the human check of a seeded 20% subset (docs/12 §3.8)
            from sklearn.metrics import cohen_kappa_score

            hum = pd.read_csv(filled, dtype=str).dropna(subset=["human_code"])
            auto = pd.DataFrame(r["records"]).set_index("id")["code"]
            codes_kappa = cohen_kappa_score(hum["human_code"], auto.loc[hum["id"]])
            PH_FILES.append((f"kappa {m} {key}", "PR", f"Cohen's kappa auto vs human coding: {codes_kappa:.3f} (n={len(hum)})"))
    if codes:
        cd = pd.DataFrame(codes)
        ph_save((cd.groupby(["setting", "model", "cell", "code"]).size() / cd.groupby(["setting", "model", "cell"]).size())
                .rename("share").reset_index(), "xai_self_explanation_codes", "PR",
                "Coded self-explanations for false negatives and false positives (model claims, not faithful explanations)")

In [ ]:
# 2.9 EXIST test export to EvALL (PH): only once the organisers confirm English-only scoring
if not PH["evall_confirmed"]:
    print("EvALL export skipped: English-only scoring not confirmed by the organisers (docs/12 §2.9)")
else:
    out_dir, man = PH_DIR / "evall_submissions", []
    for task in ("task_1_1", "task_1_2", "task_1_3"):
        schema = get_schema("exist", task)
        for m in MODELS:
            for variant, t in (("omit", T), ("forced", None)):
                recs = REC[("exist", task, t)][m]
                parents = REC[("exist", "task_1_1", t)][m] if task == "task_1_3" else None
                for soft in (False, True):
                    items = evall_items(recs, schema, soft, parents)
                    run_id = f"{m}-{variant}-a1"
                    path = write_evall(items, out_dir / task, task, soft, PH["evall_team"], run_id)
                    man.append({"file": str(path.relative_to(PH_DIR)), "model": m, "task": task, "variant": variant,
                                "type": "soft" if soft else "hard", "n_items": len(items), "sha256": sha256_file(path)})
    ph_save(man, "evall_manifest_a1", "PH", "EvALL files with addendum naming (omit / answer-forced) and 1.3 hierarchy")

In [ ]:
# Save: run README listing every post-hoc table with its PH/PR label, environment, and a sanity verdict
readme = [f"# {PH_RUN}", "", f"**Purpose.** Post-hoc (PH) and pre-registered (PR) analyses of addendum A1 "
          f"(`{PH['addendum']}`), run in `notebooks/99_final_eval.ipynb` after the frozen sections 1-12.",
          f"**Inputs.** Frozen batch via `{PH['experiments']['frozen']}`; new experiments: "
          + ", ".join(f"{k} = {v}" for k, v in PH["experiments"].items() if k != "frozen") + ".",
          f"**Config.** B = {NB} (small tables {NB_S}), seed {PH['seed']}, primary non-answer cutoff {T}; sanity {SAN}.",
          f"**Tables** (`{PH_TABLES}`):", ""]
readme += [f"- `{n}.csv` [{lab}]: {note}" for n, lab, note in PH_FILES]
(PH_DIR / "README.md").write_text("\n".join(readme) + "\n")
(PH_RES / "README.md").write_text("\n".join(readme) + "\n")
(PH_DIR / "config.json").write_text(json.dumps(PH, indent=2, default=str))
(PH_DIR / "environment.json").write_text(json.dumps(environment_info(PROJECT_ROOT), indent=1))
print(f"{len(PH_FILES)} post-hoc tables written to {PH_TABLES}")
if SAN:
    print("SANITY OK" if len(PH_FILES) >= 15 else "SANITY FAILED: fewer tables than expected")

## Post-hoc findings (addendum A1)
TODO after the real run: summarise each PH/PR table with its source file, the status label and the numbers that matter; no number here before the run.